# 🔭 Exoplanet Transit Detection Pipeline
**Thesis: Exoplanet Transit Detection Using Traditional ML and 1D Convolutional Deep Networks**

> **Note on evaluation:** The pipeline uses GroupShuffleSplit so planets from the same host star cannot appear in both train and test sets. On a small curated demo dataset (≤ 20 targets), perfect scores indicate well-separated classes — not generalisation to the full catalog.  
IUBAT · CSC 488

### Pipeline steps
1. Install dependencies
2. Download Kepler / TESS light curves from NASA MAST
3. Preprocessing (Savitzky–Golay filter → normalize → fixed length)
4. Multi-model ML comparison: Logistic Regression, Decision Tree, KNN, SVM, Random Forest, Extra Trees, MLP, XGBoost, LightGBM
5. Main model: 1D CNN + Grad-CAM visualizations
6. Export results as JSON → load into the HTML prototype

### Usage
1. Upload this notebook to [Google Colab](https://colab.research.google.com) and run all cells top-to-bottom
2. After Cell 10 finishes, JSON files will auto-download to your computer
3. Open `exoplanet_prototype.html` in Chrome and drag the JSON files into the sidebar

**Runtime:** Select `Runtime → Change runtime type → T4 GPU` for faster CNN training.


---
## ⚙️ Cell 1 — Install dependencies

In [ ]:
!pip install -q lightkurve xgboost shap lightgbm imbalanced-learn optuna "fsspec>=2023.1.0,<=2025.3.0"
print('All packages installed')


---
## 📁 Cell 2 — (Optional) Mount Google Drive to save results

In [ ]:
USE_DRIVE = False   # ← set True to save results to your Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/exoplanet_results'
    MODELS_DIR  = '/content/drive/MyDrive/exoplanet_models'
else:
    RESULTS_DIR = '/content/results'
    MODELS_DIR  = '/content/models'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
print(f'Results → {RESULTS_DIR}')
print(f'Models  → {MODELS_DIR}')

---
## 🎯 Cell 3 — Configure your stars

Edit `KOI_IDS` and/or `TIC_IDS` with the stars you want to analyse.

In [ ]:
# ── Kepler KOI IDs ──────────────────────────────────────────────────────────
# Labels (CONFIRMED / FALSE POSITIVE) are fetched live from NASA at runtime.
# Maximise diversity: hot Jupiters, sub-Neptunes, rocky planets, multi-planet
# systems, and a rich false-positive set for a balanced, generalisable model.
#
KOI_IDS = [
    # ── Confirmed planets ───────────────────────────────────────────────────
    # Hot Jupiters / inflated giants
    'K00001.01',   # Kepler-1b   — hot Jupiter benchmark
    'K00002.01',   # Kepler-2b   — TrES-2b
    'K00003.01',   # Kepler-3b   — HAT-P-11b, Neptune-class
    'K00007.01',   # Kepler-4b
    'K00013.01',   # Kepler-13Ab — very hot Jupiter
    'K00017.01',   # Kepler-6b
    'K00018.01',   # Kepler-5b
    'K00020.01',   # Kepler-12b  — inflated hot Jupiter
    'K00022.01',   # Kepler-10b  — rocky super-Earth
    'K00098.01',   # Kepler-14b  — giant

    # Kepler-100 system (3 planets)
    'K00041.01',   # Kepler-100b
    'K00041.02',   # Kepler-100c
    'K00041.03',   # Kepler-100d

    # Kepler-20 system (5 planets)
    'K00069.01',   # Kepler-20b
    'K00069.02',   # Kepler-20c
    'K00069.03',   # Kepler-20d
    'K00070.01',   # Kepler-20e
    'K00070.02',   # Kepler-20f

    # Kepler-10c / Mega-Earth
    'K00072.01',   # Kepler-10c

    # Kepler-102 system (5 planets)
    'K00082.01',   # Kepler-102b
    'K00082.02',   # Kepler-102c
    'K00082.03',   # Kepler-102d
    'K00082.04',   # Kepler-102e
    'K00082.05',   # Kepler-102f

    # Kepler-65 system (3 planets)
    'K00085.01',   # Kepler-65b
    'K00085.02',   # Kepler-65c
    'K00085.03',   # Kepler-65d

    # Kepler-89 system (4 planets) — diverse sizes, good FP discriminator
    'K00094.01',   # Kepler-89b
    'K00094.02',   # Kepler-89c
    'K00094.03',   # Kepler-89d  (Saturn-mass)
    'K00094.04',   # Kepler-89e

    # Kepler-105 system (3 planets)
    'K00115.01',   # Kepler-105b
    'K00115.02',   # Kepler-105c
    'K00115.03',   # Kepler-105d

    # Kepler-48 system (3 planets)
    'K00148.01',   # Kepler-48b
    'K00148.02',   # Kepler-48c
    'K00148.03',   # Kepler-48d

    # Kepler-25 / 36 / 37 systems
    'K00244.01',   # Kepler-25b
    'K00244.02',   # Kepler-25c
    'K00245.01',   # Kepler-37b  — sub-Mercury
    'K00277.01',   # Kepler-36b
    'K00277.02',   # Kepler-36c

    # Kepler-90 system (7 planets! — largest confirmed multi-planet system)
    'K00351.01',   # Kepler-90b
    'K00351.02',   # Kepler-90c
    'K00351.03',   # Kepler-90d
    'K00351.04',   # Kepler-90e
    'K00351.05',   # Kepler-90f
    'K00351.06',   # Kepler-90g
    'K00351.07',   # Kepler-90h

    # Kepler-103 system
    'K00408.01',   # Kepler-103b
    'K00408.02',   # Kepler-103c

    # Misc confirmed
    'K00520.01',   # Kepler-174b

    # Kepler-186 system (5 planets, HZ planet)
    'K00571.01',   # Kepler-186b
    'K00571.02',   # Kepler-186c
    'K00571.03',   # Kepler-186d
    'K00571.04',   # Kepler-186e
    'K00571.05',   # Kepler-186f  — habitable zone!

    # Kepler-62 system (5 planets, 2 in HZ)
    'K00701.01',   # Kepler-62b
    'K00701.02',   # Kepler-62c
    'K00701.03',   # Kepler-62d
    'K00701.04',   # Kepler-62e  — HZ
    'K00701.05',   # Kepler-62f  — HZ

    # Kepler-201 / 177
    'K00755.01',   # Kepler-201b
    'K00975.01',   # Kepler-177b

    # Kepler-32 system (5-planet compact system)
    'K00952.01',   # Kepler-32b
    'K00952.02',   # Kepler-32c
    'K00952.03',   # Kepler-32d
    'K00952.04',   # Kepler-32e
    'K00952.05',   # Kepler-32f

    # ── Known false positives ───────────────────────────────────────────────
    # Original FP set
    'K00005.01',
    'K00012.01',
    'K00019.01',
    'K00024.01',
    'K00028.01',
    'K00034.01',
    'K00042.01',
    'K00049.01',
    'K00117.01',
    'K00122.01',
    'K00137.01',
    'K00152.01',
    'K00165.01',
    'K00175.01',
    'K00191.01',
    'K00218.01',
    'K00228.01',
    'K00256.01',
    'K00305.01',
    'K00143.01',
    'K00150.01',
    'K00162.01',
    'K00174.01',
    'K00188.01',
    'K00207.01',
    # Additional FPs (diverse eclipsing binary / background blend scenarios)
    'K00029.01',
    'K00031.01',
    'K00040.01',
    'K00043.01',
    'K00053.01',
    'K00060.01',
    'K00064.01',
    'K00073.01',
    'K00076.01',
    'K00084.01',
    'K00088.01',
    'K00099.01',
    'K00103.01',
    'K00108.01',
]

# ── TESS TIC IDs ─────────────────────────────────────────────────────────────
TIC_IDS = [
    # ── Confirmed TESS planets (TOIs) ────────────────────────────────────────
    # TOI-270 system (3 confirmed sub-Neptunes, benchmark TESS system)
    '307210830',   # TOI-270b — inner sub-Neptune
    '307210830',   # TOI-270c — shared TIC, queried per-planet via period
    # TOI-700 (habitable zone Earth-size planet)
    '150428135',   # TOI-700 b, c, d — HZ confirmed
    # TOI-178 (Laplace resonance chain, 6 planets)
    '92352620',    # TOI-178 system
    # TOI-421 (warm sub-Saturn + hot Neptune)
    '44792069',    # TOI-421b and c
    # TOI-125 (3 sub-Neptunes)
    '265980385',   # TOI-125b, c, d
    # TOI-776 (2 sub-Neptunes in HZ)
    '32769860',    # TOI-776b, c
    # TOI-1431 (ultra-hot Jupiter)
    '55525572',    # TOI-1431b
    # TOI-1634 (super-Earth)
    '261136679',   # TOI-1634b
    # TOI-1452 (ocean world candidate)
    '264922395',   # TOI-1452b
    # ── TESS false positives ──────────────────────────────────────────────────
    '207141131',   # eclipsing binary FP
    '219854474',   # background EB FP
    '8750009',     # grazing eclipsing binary FP
    '273185330',   # FP — stellar variability
    '362249359',   # FP — instrumental artifact
]

# ── Preprocessing settings ───────────────────────────────────────────────────
APPLY_SG_FILTER = True
NORMALIZE_FLUX  = True
SEQ_LEN         = 2000

# ── CNN training settings ─────────────────────────────────────────────────────
CNN_EPOCHS     = 80    # more epochs — early stopping guards against overfit
CNN_BATCH_SIZE = 16
CNN_LR         = 5e-4

print(f'Kepler targets: {len(KOI_IDS)}  |  TESS targets: {len(TIC_IDS)}')
print(f'SEQ_LEN={SEQ_LEN}  CNN_EPOCHS={CNN_EPOCHS}  CNN_LR={CNN_LR}')


---
## 📥 Cell 4 — Download light curves from NASA MAST

In [ ]:
import warnings, requests, json, io
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.stats import skew, kurtosis
import lightkurve as lk
warnings.filterwarnings('ignore')

NASA_TAP  = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync'
CUMUL_URL = (NASA_TAP + '?query=SELECT+kepoi_name,kepid,koi_pdisposition,'
             'koi_period,koi_duration,koi_depth,koi_prad,koi_sma,koi_teq,'
             'koi_insol,koi_impact,koi_srad,koi_smass,koi_steff,koi_slogg,'
             'koi_kepmag,koi_score,ra,dec+FROM+cumulative&format=csv')
TOI_URL   = (NASA_TAP + '?query=SELECT+tid,tfopwg_disp,pl_orbper,pl_trandurh,'
              'pl_trandep,pl_rade,pl_orbsmax,pl_eqt,pl_insol,pl_imppar,'
              'st_rad,st_mass,st_teff,st_logg,st_tmag,ra,dec+FROM+toi&format=csv')

# Download full tables once — much more reliable than per-star API calls
print('Downloading cumulative KOI table...', end=' ', flush=True)
try:
    koi_df = pd.read_csv(io.StringIO(requests.get(CUMUL_URL, timeout=60).text))
    koi_df.columns = [c.strip() for c in koi_df.columns]
    koi_df['kepoi_name'] = koi_df['kepoi_name'].str.strip()
    koi_df = koi_df.set_index('kepoi_name')
    print(f'OK  {len(koi_df)} rows')
except Exception as e:
    print(f'WARN  {e}  (falling back to per-star API)')
    koi_df = None

print('Downloading TOI table...', end=' ', flush=True)
try:
    toi_df = pd.read_csv(io.StringIO(requests.get(TOI_URL, timeout=60).text))
    toi_df.columns = [c.strip() for c in toi_df.columns]
    toi_df['tid'] = toi_df['tid'].astype(str).str.strip()
    toi_df = toi_df.set_index('tid')
    print(f'OK  {len(toi_df)} rows')
except Exception as e:
    print(f'WARN  {e}')
    toi_df = None

def koi_to_kic(koi_id):
    try:
        if koi_df is not None and koi_id in koi_df.index:
            return str(int(koi_df.loc[koi_id, 'kepid']))
    except: pass
    try:
        q = f"SELECT kepid FROM cumulative WHERE kepoi_name='{koi_id}'"
        r = requests.get(NASA_TAP, params={'query': q, 'format': 'json'}, timeout=15)
        data = r.json()
        if data: return str(data[0]['kepid'])
    except: pass
    # Hard fallback: query API individually
    try:
        q2 = f"SELECT kepid FROM cumulative WHERE kepoi_name='{koi_id}'"
        r2 = requests.get(NASA_TAP, params={'query': q2, 'format': 'json'}, timeout=20)
        data2 = r2.json()
        if data2: return str(data2[0]['kepid'])
    except: pass
    raise ValueError(f'Could not resolve KIC for {koi_id} — check the KOI ID format')

def fetch_koi_period_and_label(koi_id):
    period, label = 5.0, 'unknown'
    try:
        if koi_df is not None and koi_id in koi_df.index:
            row    = koi_df.loc[koi_id]
            period = float(row.get('koi_period') or 5.0)
            disp   = str(row.get('koi_pdisposition') or '').upper()
        else:
            q = f"SELECT koi_period,koi_pdisposition FROM cumulative WHERE kepoi_name='{koi_id}'"
            r = requests.get(NASA_TAP, params={'query': q, 'format': 'json'}, timeout=15)
            data = r.json()
            if not data: return period, label
            row    = data[0]
            period = float(row.get('koi_period') or 5.0)
            disp   = str(row.get('koi_pdisposition') or '').upper()
        if 'CANDIDATE' in disp or 'CONFIRMED' in disp: label = 'planet'
        elif 'FALSE' in disp: label = 'false_positive'
    except Exception as e:
        print(f'  Archive query failed: {e}')
    return period, label

def fetch_tic_period_and_label(tic_id):
    period, label = 5.0, 'unknown'
    try:
        if toi_df is not None and str(tic_id) in toi_df.index:
            row    = toi_df.loc[str(tic_id)]
            period = float(row.get('pl_orbper') or 5.0)
            disp   = str(row.get('tfopwg_disp') or '').upper()
        else:
            q = f"SELECT pl_orbper,tfopwg_disp FROM toi WHERE tid={tic_id}"
            r = requests.get(NASA_TAP, params={'query': q, 'format': 'json'}, timeout=15)
            data = r.json()
            if not data: return period, label
            row    = data[0]
            period = float(row.get('pl_orbper') or 5.0)
            disp   = str(row.get('tfopwg_disp') or '').upper()
        if 'PC' in disp or 'KP' in disp: label = 'planet'
        elif 'FP' in disp: label = 'false_positive'
    except Exception as e:
        print(f'  Archive query failed: {e}')
    return period, label

def phase_fold(time, flux, period, bins=2000):
    phase = (time % period) / period
    idx   = np.argsort(phase)
    ps, fs = phase[idx], flux[idx]
    edges  = np.linspace(0, 1, bins + 1)
    binned = np.array([
        np.nanmedian(fs[(ps >= edges[i]) & (ps < edges[i+1])])
        if ((ps >= edges[i]) & (ps < edges[i+1])).sum() > 0 else 1.0
        for i in range(bins)
    ])
    return (edges[:-1] + edges[1:]) / 2, binned

def download_kepler(koi_id):
    kic  = koi_to_kic(koi_id)
    col  = lk.search_lightcurve(f'KIC {kic}', mission='Kepler', cadence='long').download_all()
    if not col or len(col) == 0: raise ValueError(f'No data for KIC {kic}')
    lc   = col.stitch().normalize().remove_nans().remove_outliers(sigma=5)
    period, label = fetch_koi_period_and_label(koi_id)
    phase, flux   = phase_fold(lc.time.value, lc.flux.value, period)
    print(f'  KIC {kic} | {len(lc.time)} cadences | period={period:.3f}d | label={label}')
    return phase, flux, period, label

def download_tess(tic_id):
    col = lk.search_lightcurve(f'TIC {tic_id}', mission='TESS').download_all()
    if not col or len(col) == 0: raise ValueError(f'No data for TIC {tic_id}')
    lc  = col.stitch().normalize().remove_nans().remove_outliers(sigma=5)
    period, label = fetch_tic_period_and_label(tic_id)
    phase, flux   = phase_fold(lc.time.value, lc.flux.value, period)
    print(f'  TIC {tic_id} | {len(lc.time)} cadences | period={period:.3f}d | label={label}')
    return phase, flux, period, label

# Download all targets
all_flux, all_labels, all_ids, all_missions = [], [], [], []

for kid in KOI_IDS:
    print(f'\n[{kid}]')
    try:
        _, flux, _, label = download_kepler(kid)
        all_flux.append(flux)
        all_labels.append(1 if label == 'planet' else 0)
        all_ids.append(kid)
        all_missions.append('Kepler')
    except Exception as e:
        print(f'  ERROR: {e}')

for tid in TIC_IDS:
    print(f'\n[TIC-{tid}]')
    try:
        _, flux, _, label = download_tess(tid)
        all_flux.append(flux)
        all_labels.append(1 if label == 'planet' else 0)
        all_ids.append(f'TIC-{tid}')
        all_missions.append('TESS')
    except Exception as e:
        print(f'  ERROR: {e}')

print(f'\nDownloaded {len(all_flux)} light curves  |  Planets: {sum(all_labels)}  |  Non-planets: {len(all_labels)-sum(all_labels)}')


---
## 🔧 Cell 5 — Preprocessing

In [ ]:
import matplotlib.pyplot as plt
import io, base64

def fig_to_b64(fig, dpi=80):
    """Convert a matplotlib figure to a base64 data-URI string."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode('utf-8')
    buf.close()
    return 'data:image/png;base64,' + b64

def preprocess(flux, apply_sg=APPLY_SG_FILTER, normalize=NORMALIZE_FLUX, target_len=SEQ_LEN):
    nans = np.isnan(flux)
    if nans.any():
        x = np.arange(len(flux))
        flux = np.interp(x, x[~nans], flux[~nans])
    if apply_sg and len(flux) > 51:
        trend = savgol_filter(flux, window_length=51, polyorder=3)  # 51 phase-space bins (not cadences)
        flux  = flux / trend
    if normalize:
        med = np.nanmedian(flux)
        if med != 0: flux = flux / med
    if len(flux) != target_len:
        flux = np.interp(np.linspace(0, 1, target_len), np.linspace(0, 1, len(flux)), flux)
    return flux.astype(np.float32)

processed = [preprocess(f) for f in all_flux]
print(f'✓ Preprocessed {len(processed)} light curves → shape {processed[0].shape}')

# ── Plot up to 6 light curves ─────────────────────────────────────────────────
n_plot = min(6, len(processed))
cols   = min(3, n_plot)
rows   = (n_plot + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3 * rows))
axes_flat = axes.flat if hasattr(axes, 'flat') else [axes]
fig.patch.set_facecolor('#0b0f1a')
for i, ax in enumerate(axes_flat):
    if i < n_plot:
        phase = np.linspace(0, 1, SEQ_LEN)
        ax.plot(phase, processed[i], lw=0.8, color='#3b82f6', alpha=0.9)
        lbl = 'Planet' if all_labels[i] == 1 else 'Non-planet'
        ax.set_title(f'{all_ids[i]}  ({lbl})', fontsize=9, color='white')
        ax.set_xlabel('Phase', fontsize=8, color='gray')
        ax.set_ylabel('Norm. Flux', fontsize=8, color='gray')
        ax.set_facecolor('#111827')
        ax.tick_params(colors='gray')
        ax.spines[:].set_color('#1e2d47')
        ax.axvspan(0.45, 0.55, alpha=0.12, color='#10b981', label='Transit window')
    else:
        ax.axis('off')
plt.suptitle('Preprocessed Phase-Folded Light Curves', color='white', fontsize=12, y=1.01)
plt.tight_layout()
preprocess_b64 = fig_to_b64(fig, dpi=90)
plt.savefig(f'{RESULTS_DIR}/preprocessed_curves.png', dpi=100, bbox_inches='tight',
            facecolor='#0b0f1a')
plt.show()
print(f'✓ Plot saved ({len(preprocess_b64)//1024} KB base64)')


---
## Cell 6 — Multi-Model Comparison + SHAP + Explainability Metrics

Trains 7 classifiers on tabular features extracted from phase-folded light curves.

**Models**: Logistic Regression, Decision Tree, KNN, Random Forest, Extra Trees, XGBoost, LightGBM (if installed)

**Explainability metrics**:
- **SHAP Faithfulness**: accuracy drop when top-3 SHAP features are masked — higher means the explanation is trustworthy
- **SHAP Sparsity**: fraction of near-zero SHAP values — higher means fewer features drive the decision
- **SHAP Consistency**: mean Spearman rank correlation of feature importance across samples — near 1.0 = stable
- **Grad-CAM Localization**: fraction of CNN activation inside the expected transit window (computed in Cell 8)

In [ ]:
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GroupShuffleSplit, GroupKFold,
                                     cross_val_score, RandomizedSearchCV,
                                     cross_val_predict)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             average_precision_score, balanced_accuracy_score,
                             confusion_matrix, roc_curve)
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              VotingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from xgboost import XGBClassifier
import shap, matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis, spearmanr
from scipy.signal import periodogram

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    print('optuna not found — falling back to RandomizedSearchCV')

# ── 30-feature extractor ──────────────────────────────────────────────────────
def extract_features(flux):
    N    = len(flux)
    med  = float(np.median(flux))
    mn   = float(np.mean(flux))
    phase_arr = np.linspace(0, 1, N)

    # Transit identification
    depth  = float(med - np.min(flux))
    thresh = med - 0.5 * depth
    in_tr  = flux < thresh

    # Duration
    duration = float(in_tr.sum() / N)

    # Out-of-transit flux
    out_flux = flux[~in_tr] if (~in_tr).sum() > 3 else flux
    noise    = float(np.std(out_flux))
    snr      = float(depth / noise) if noise > 0 else 0.0

    # Even/odd diagnostic (eclipsing binaries show alternating depths)
    mid       = N // 2
    even_d    = float(med - np.min(flux[:mid]))
    odd_d     = float(med - np.min(flux[mid:]))
    even_odd_diff = float(abs(even_d - odd_d))

    # Secondary eclipse — primary window (phase 0.45–0.55)
    s0, s1    = int(0.45 * N), int(0.55 * N)
    sec_reg   = flux[s0:s1]
    sec_depth = float(med - np.min(sec_reg)) if len(sec_reg) > 0 else 0.0
    sec_ratio = float(sec_depth / depth) if depth > 0 else 0.0

    # Flat-bottom fraction
    flat_frac = float(np.sum(flux[in_tr] < (med - 0.8 * depth)) / max(in_tr.sum(), 1))

    # Transit symmetry
    in_idx = np.where(in_tr)[0]
    if len(in_idx) > 4:
        half     = len(in_idx) // 2
        symmetry = float(abs(np.mean(flux[in_idx[:half]]) - np.mean(flux[in_idx[half:]])))
    else:
        symmetry = 0.0

    # Limb darkening proxy
    if in_tr.sum() > 6:
        xi       = np.linspace(-1, 1, in_tr.sum())
        coeffs   = np.polyfit(xi, flux[in_tr], 2)
        ld_proxy = float(abs(coeffs[0]))
    else:
        ld_proxy = 0.0

    # Out-of-transit range
    oot_range = float(np.percentile(out_flux, 99) - np.percentile(out_flux, 1)) \
                if len(out_flux) > 5 else 0.0

    # Scatter ratio & V-shape index
    in_scatter  = float(np.std(flux[in_tr])) if in_tr.sum() > 3 else noise
    out_scatter = float(np.std(out_flux))     if len(out_flux) > 3 else noise
    scatter_ratio = float(in_scatter / max(out_scatter, 1e-9))
    if in_tr.sum() > 3:
        in_mean  = np.mean(flux[in_tr])
        in_min   = np.min(flux[in_tr])
        vshape   = float(abs(in_mean - in_min) / max(depth, 1e-9))
    else:
        vshape   = 0.0

    fold_quality      = float(np.var(flux))
    c_skew            = float(skew(flux))   # flux-distribution skewness (proxy for photometric asymmetry; not photocenter motion)
    kurt              = float(kurtosis(flux))
    depth_snr         = float(depth / max(np.std(flux), 1e-8))

    # ── NEW features (22→30) ─────────────────────────────────────────────────
    # 1. Bottom flatness: std of the deepest 50% of in-transit pts / depth
    #    Low = flat bottom = planet; high = V-shape = binary
    if len(in_idx) > 4:
        in_flux_pts = flux[in_idx]
        deep_thresh = np.percentile(in_flux_pts, 50)
        deep_pts    = in_flux_pts[in_flux_pts <= deep_thresh]
        bottom_flatness = float(np.std(deep_pts) / max(depth, 1e-9)) \
                          if len(deep_pts) > 1 else 0.0
    else:
        bottom_flatness = 0.0

    # 2 & 3. Secondary eclipses at phases 0.27–0.33 and 0.67–0.73
    #        (eccentric binaries show a secondary at a shifted phase)
    s03 = flux[int(0.27*N):int(0.33*N)]
    s07 = flux[int(0.67*N):int(0.73*N)]
    sec03_ratio = float((med - np.min(s03)) / max(depth, 1e-9)) if len(s03) > 0 else 0.0
    sec07_ratio = float((med - np.min(s07)) / max(depth, 1e-9)) if len(s07) > 0 else 0.0

    # 4. Half-period depth: eclipse near phase 0.0 (equal-depth EB indicator)
    hp_reg = np.concatenate([flux[:int(0.05*N)], flux[int(0.95*N):]])
    half_period_depth = float((med - np.min(hp_reg)) / max(depth, 1e-9)) \
                        if len(hp_reg) > 0 else 0.0

    # 5. Robust SNR using Median Absolute Deviation (more resistant to outliers)
    if len(out_flux) > 5:
        mad_oot   = float(np.median(np.abs(out_flux - np.median(out_flux))))
        snr_robust = float(depth / max(1.4826 * mad_oot, 1e-9))
    else:
        snr_robust = snr

    # 6. Transit phase shift: |phase of flux minimum − 0.5|
    #    Well-folded real transit: near 0.0; poorly-aligned EB: higher
    transit_phase_shift = float(abs(phase_arr[int(np.argmin(flux))] - 0.5))

    # 7. Outlier fraction: fraction of OOT points more than 3σ from median
    #    Spotted/variable stars produce more outliers → likely FP
    if len(out_flux) > 5:
        oot_med  = float(np.median(out_flux))
        oot_sig  = float(np.std(out_flux))
        outlier_fraction = float(np.mean(np.abs(out_flux - oot_med) > 3 * oot_sig))
    else:
        outlier_fraction = 0.0

    # 8. Ingress proxy: mean absolute slope at the two transit edges
    #    Steep = V-shape = binary; gradual = box = planet
    if len(in_idx) > 6:
        n_edge = max(2, len(in_idx) // 6)
        l_idx  = in_idx[:n_edge]
        r_idx  = in_idx[-n_edge:]
        try:
            lslope = abs(float(np.polyfit(l_idx, flux[l_idx], 1)[0]))
            rslope = abs(float(np.polyfit(r_idx, flux[r_idx], 1)[0]))
            ingress_proxy = float((lslope + rslope) / 2 / max(depth, 1e-9))
        except Exception:
            ingress_proxy = 0.0
    else:
        ingress_proxy = 0.0

    return {
        # Original 22
        'transit_depth':     depth,
        'transit_duration':  duration,
        'snr':               snr,
        'even_odd_diff':     even_odd_diff,
        'centroid_skew':     c_skew,
        'kurtosis':          kurt,
        'std':               float(np.std(flux)),
        'mean':              mn,
        'p5_p95_range':      float(np.percentile(flux, 95) - np.percentile(flux, 5)),
        'secondary_depth':   sec_depth,
        'secondary_ratio':   sec_ratio,
        'flat_fraction':     flat_frac,
        'transit_symmetry':  symmetry,
        'oot_range':         oot_range,
        'depth_snr':         depth_snr,
        'transit_count':     float(in_tr.sum()),
        'ld_proxy':          ld_proxy,
        'scatter_ratio':     scatter_ratio,
        'vshape_index':      vshape,
        'fold_quality':      fold_quality,
        'in_out_noise_diff': float(abs(in_scatter - out_scatter)),
        'transit_fill':      float(in_tr.sum() / N),
        # New 8
        'bottom_flatness':   bottom_flatness,
        'sec03_ratio':       sec03_ratio,
        'sec07_ratio':       sec07_ratio,
        'half_period_depth': half_period_depth,
        'snr_robust':        snr_robust,
        'transit_phase_shift': transit_phase_shift,
        'outlier_fraction':  outlier_fraction,
        'ingress_proxy':     ingress_proxy,
    }

feat_names = list(extract_features(processed[0]).keys())
X_feat = np.array([list(extract_features(f).values()) for f in processed])
X_raw  = np.array(processed)
y      = np.array(all_labels)

# ── Group labels: same host star = same group (prevent leakage in CV) ─────────
def _koi_group(sid):
    """'K00351.01' → 'K00351';  'TIC-394137592' → 'TIC-394137592'"""
    if isinstance(sid, str) and sid.startswith('K') and '.' in sid:
        return sid.split('.')[0]
    return str(sid)
_ids_for_groups = all_ids[:len(y)] if len(all_ids) >= len(y) else [str(i) for i in range(len(y))]
groups = np.array([_koi_group(sid) for sid in _ids_for_groups])
unique_groups = np.unique(groups)
print(f'Host-star groups: {len(unique_groups)} (used for leakage-free split and CV)')

print(f'Features: {len(feat_names)} → {feat_names}')

# ── Class balance ─────────────────────────────────────────────────────────────
unique_classes = np.unique(y)
if len(unique_classes) < 2:
    raise ValueError(
        f'Only class {unique_classes} found — add more targets to KOI_IDS.\n'
        'Need at least one confirmed planet AND one false positive.')

n_pos       = max(1, int(y.sum()))
n_neg       = max(1, int((y == 0).sum()))
pos_weight  = float(n_neg) / float(n_pos)
print(f'Planets: {n_pos}  |  Non-planets: {n_neg}  |  pos_weight: {pos_weight:.2f}')

# ── 3-way group-based split: train / val / held-out test ────────────────────
# §11: Test set is COMPLETELY held out — not used for tuning, threshold
#       selection, or feature selection. Final evaluation only.
# ⚠ Small-dataset notice: with < 20 labeled targets, perfect scores (1.000)
# reflect the well-separated nature of the curated demo set — NOT generalisation
# to the full Kepler/TESS catalogs. Extend the KOI/TIC lists for rigorous evaluation.
small_batch  = len(y) < 8
n_g = len(np.unique(groups))
idx_all = np.arange(len(y))
if small_batch or n_g < 4:
    print(f'Warning: too few samples/groups ({len(y)}/{n_g}) — using single split.')
    _gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    _tv, val = next(_gss.split(X_feat, y, groups))
    tr, test = _tv, np.array([], dtype=int)
else:
    # Step 1: carve out 15% held-out test set (group-aware)
    _gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=99)
    _tv, test = next(_gss1.split(X_feat, y, groups))
    # Step 2: from remaining, split 80/20 for train/val (group-aware)
    _gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    _tr_idx, _val_idx = next(_gss2.split(X_feat[_tv], y[_tv], groups[_tv]))
    tr  = _tv[_tr_idx]
    val = _tv[_val_idx]

print(f'Train: {len(tr)} ({int(y[tr].sum())}+/{int((y[tr]==0).sum())}-) | '
      f'Val: {len(val)} ({int(y[val].sum())}+/{int((y[val]==0).sum())}-) | '
      f'Test (held-out): {len(test)} ({int(y[test].sum()) if len(test) else 0}+/{int((y[test]==0).sum()) if len(test) else 0}-)')
if len(test):
    print(f'Test groups: {np.unique(groups[test]).tolist()}')
print('⚠ Test set is locked — do not use for tuning or threshold selection.')

# ── RobustScaler ──────────────────────────────────────────────────────────────
scaler    = RobustScaler()
X_tr_sc   = scaler.fit_transform(X_feat[tr])
X_val_sc  = scaler.transform(X_feat[val])

# ── SMOTE ─────────────────────────────────────────────────────────────────────
X_tr_aug, y_tr_aug = X_feat[tr].copy(), y[tr].copy()
X_tr_sc_aug        = X_tr_sc.copy()
smote_applied      = False

if SMOTE_AVAILABLE and not small_batch:
    n_min_tr = min(int(y[tr].sum()), int((y[tr] == 0).sum()))
    if n_min_tr >= 2:
        try:
            k_sm = max(1, min(n_min_tr - 1, 3))
            sm   = SMOTE(k_neighbors=k_sm, random_state=42)
            X_tr_sc_aug, y_tr_aug = sm.fit_resample(X_tr_sc, y[tr])
            X_tr_aug, _           = sm.fit_resample(X_feat[tr], y[tr])
            smote_applied = True
            print(f'SMOTE: {len(y[tr])} → {len(y_tr_aug)} training samples')
        except Exception as e:
            print(f'SMOTE skipped: {e}')

# ── Hyperparameter tuning helpers ─────────────────────────────────────────────
cv_tune = StratifiedKFold(n_splits=min(3, max(2, min(n_pos, n_neg))),
                           shuffle=True, random_state=42)

def tune_model_rs(model, param_dist, X, y_, n_iter=15):
    """RandomizedSearchCV fallback."""
    if len(np.unique(y_)) < 2 or len(y_) < 6:
        model.fit(X, y_); return model
    search = RandomizedSearchCV(model, param_dist, n_iter=n_iter,
                                scoring='f1', cv=cv_tune,
                                random_state=42, n_jobs=-1, refit=True, error_score=0.0)
    search.fit(X, y_)
    print(f'  RS best F1={search.best_score_:.3f}  params={search.best_params_}')
    return search.best_estimator_

def tune_xgb_optuna(X, y_, n_trials=25):
    """Optuna Bayesian search for XGBoost."""
    def objective(trial):
        params = {
            'max_depth':         trial.suggest_int('max_depth', 3, 7),
            'learning_rate':     trial.suggest_float('lr', 0.005, 0.1, log=True),
            'n_estimators':      trial.suggest_int('n_estimators', 200, 600),
            'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree':  trial.suggest_float('col_bt', 0.5, 1.0),
            'reg_alpha':         trial.suggest_float('alpha', 0.0, 1.0),
            'reg_lambda':        trial.suggest_float('lambda', 0.5, 3.0),
            'min_child_weight':  trial.suggest_int('mcw', 1, 7),
            'gamma':             trial.suggest_float('gamma', 0.0, 0.5),
            'scale_pos_weight':  pos_weight,
            'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1,
        }
        clf = XGBClassifier(**params)
        if len(np.unique(y_)) < 2 or len(y_) < 6:
            clf.fit(X, y_)
            return float(f1_score(y_, clf.predict(X), zero_division=0))
        scores = cross_val_score(clf, X, y_, cv=cv_tune, scoring='f1', error_score=0.0)
        return float(scores.mean())

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    best_model = XGBClassifier(
        max_depth=best['max_depth'],
        learning_rate=best['lr'],
        n_estimators=best['n_estimators'],
        subsample=best['subsample'],
        colsample_bytree=best['col_bt'],
        reg_alpha=best['alpha'],
        reg_lambda=best['lambda'],
        min_child_weight=best['mcw'],
        gamma=best['gamma'],
        scale_pos_weight=pos_weight,
        eval_metric='logloss', random_state=42, n_jobs=-1,
    )
    best_model.fit(X, y_)
    print(f'  Optuna XGB best F1={study.best_value:.3f}  params={best}')
    return best_model

def tune_lgbm_optuna(X, y_, n_trials=25):
    from lightgbm import LGBMClassifier
    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 200, 600),
            'max_depth':         trial.suggest_int('max_depth', 3, 7),
            'learning_rate':     trial.suggest_float('lr', 0.005, 0.1, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 15, 63),
            'min_child_samples': trial.suggest_int('mcs', 2, 10),
            'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree':  trial.suggest_float('col_bt', 0.5, 1.0),
            'class_weight':      'balanced',
            'verbose': -1, 'random_state': 42,
        }
        clf = LGBMClassifier(**params)
        if len(np.unique(y_)) < 2 or len(y_) < 6:
            clf.fit(X, y_)
            return float(f1_score(y_, clf.predict(X), zero_division=0))
        scores = cross_val_score(clf, X, y_, cv=cv_tune, scoring='f1', error_score=0.0)
        return float(scores.mean())

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    best_model = LGBMClassifier(
        n_estimators=best['n_estimators'], max_depth=best['max_depth'],
        learning_rate=best['lr'], num_leaves=best['num_leaves'],
        min_child_samples=best['mcs'], subsample=best['subsample'],
        colsample_bytree=best['col_bt'],
        class_weight='balanced', verbose=-1, random_state=42,
    )
    best_model.fit(X, y_)
    print(f'  Optuna LGBM best F1={study.best_value:.3f}  params={best}')
    return best_model

# ── Tune base learners ────────────────────────────────────────────────────────
xgb_base = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.02,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
    scale_pos_weight=pos_weight, reg_alpha=0.2, reg_lambda=2.0,
    gamma=0.05, eval_metric='logloss', random_state=42, n_jobs=-1)

rf_base = RandomForestClassifier(
    n_estimators=400, max_depth=None, max_features='sqrt',
    min_samples_leaf=1, class_weight='balanced_subsample',
    bootstrap=True, oob_score=True, random_state=42, n_jobs=-1)

et_base = ExtraTreesClassifier(
    n_estimators=400, max_depth=None, max_features='sqrt',
    class_weight='balanced_subsample', random_state=42, n_jobs=-1)

if OPTUNA_AVAILABLE and not small_batch:
    print('\nTuning XGBoost with Optuna (25 trials)...')
    xgb_tuned = tune_xgb_optuna(X_tr_aug, y_tr_aug, n_trials=25)
    print('Tuning Random Forest with RandomizedSearchCV...')
    rf_tuned  = tune_model_rs(rf_base, {
        'n_estimators': [300, 400, 600], 'max_depth': [None, 8, 12],
        'max_features': ['sqrt', 'log2', 0.5], 'min_samples_leaf': [1, 2, 3],
    }, X_tr_aug, y_tr_aug)
    print('Tuning Extra Trees with RandomizedSearchCV...')
    et_tuned  = tune_model_rs(et_base, {
        'n_estimators': [300, 400, 600], 'max_features': ['sqrt', 'log2', 0.5],
        'min_samples_leaf': [1, 2],
    }, X_tr_aug, y_tr_aug)
else:
    print('\nTuning XGBoost with RandomizedSearchCV...')
    xgb_tuned = tune_model_rs(xgb_base, {
        'max_depth': [4,5,6], 'learning_rate': [0.01,0.02,0.05],
        'n_estimators': [300,400,500], 'subsample': [0.7,0.8,0.9],
        'colsample_bytree': [0.6,0.7,0.8], 'reg_alpha': [0.1,0.2,0.5],
        'min_child_weight': [2,3,5],
    }, X_tr_aug, y_tr_aug)
    print('Tuning Random Forest...')
    rf_tuned  = tune_model_rs(rf_base, {
        'n_estimators': [300,400,600], 'max_depth': [None,8,12],
        'max_features': ['sqrt','log2',0.5], 'min_samples_leaf': [1,2,3],
    }, X_tr_aug, y_tr_aug)
    print('Tuning Extra Trees...')
    et_tuned  = tune_model_rs(et_base, {
        'n_estimators': [300,400,600], 'max_features': ['sqrt','log2',0.5],
        'min_samples_leaf': [1,2],
    }, X_tr_aug, y_tr_aug)

lgbm_tuned = None
try:
    from lightgbm import LGBMClassifier
    if OPTUNA_AVAILABLE and not small_batch:
        print('Tuning LightGBM with Optuna (25 trials)...')
        lgbm_tuned = tune_lgbm_optuna(X_tr_aug, y_tr_aug, n_trials=25)
    else:
        lgbm_base = LGBMClassifier(n_estimators=400, max_depth=5, learning_rate=0.02,
                                    class_weight='balanced', min_child_samples=3,
                                    verbose=-1, random_state=42)
        print('Tuning LightGBM with RandomizedSearchCV...')
        lgbm_tuned = tune_model_rs(lgbm_base, {
            'n_estimators': [300,400,500], 'max_depth': [4,5,6],
            'learning_rate': [0.01,0.02,0.05], 'num_leaves': [15,31,63],
            'min_child_samples': [2,3,5],
        }, X_tr_aug, y_tr_aug)
    print('LightGBM tuned')
except Exception as e:
    print(f'LightGBM skipped: {e}')

# ── Individual model configs (for leaderboard table) ─────────────────────────
k_nn = max(1, min(3, len(tr) - 1))

model_configs = {
    'Logistic Regression': (Pipeline([('scl', RobustScaler()),
                             ('clf', LogisticRegression(max_iter=3000, C=0.5,
                               class_weight='balanced', solver='saga',
                               random_state=42))]), False),
    'Decision Tree':       (DecisionTreeClassifier(max_depth=5, min_samples_leaf=2,
                              class_weight='balanced', random_state=42), False),
    'KNN':                 (Pipeline([('scl', RobustScaler()),
                             ('clf', KNeighborsClassifier(n_neighbors=k_nn,
                               weights='distance', metric='manhattan'))]), False),
    'SVM':                 (Pipeline([('scl', RobustScaler()),
                             ('clf', SVC(kernel='rbf', C=3.0, gamma='scale',
                               class_weight='balanced', probability=True,
                               random_state=42))]), False),
    'Random Forest':       (rf_tuned, False),
    'Extra Trees':         (et_tuned, False),
    'MLP':                 (Pipeline([('scl', RobustScaler()),
                             ('clf', MLPClassifier(hidden_layer_sizes=(256,128,64),
                               max_iter=800, alpha=1e-3, early_stopping=True,
                               validation_fraction=0.15, random_state=42))]), False),
    'XGBoost':             (xgb_tuned, False),
}
if lgbm_tuned is not None:
    model_configs['LightGBM'] = (lgbm_tuned, False)

# ── Expert Stacking Ensemble (replace simple VotingClassifier) ────────────────
#
# Level-1 base learners each wrapped in Pipelines that handle their own scaling.
# StackingClassifier generates k-fold cross-val predictions from each base
# learner; those become features for the LR meta-learner.
# This is strictly more powerful than soft-voting because the meta-learner
# learns *how much to trust* each base learner.
#
print('\nBuilding StackingClassifier meta-ensemble...')

stacking_estimators = [
    ('rf',  clone(rf_tuned)),
    ('et',  clone(et_tuned)),
    ('xgb', clone(xgb_tuned)),
    ('svm', Pipeline([('scl', RobustScaler()),
                      ('clf', SVC(kernel='rbf', C=3.0, gamma='scale',
                                  class_weight='balanced', probability=True,
                                  random_state=42))])),
    ('lr',  Pipeline([('scl', RobustScaler()),
                      ('clf', LogisticRegression(C=0.5, class_weight='balanced',
                                                  max_iter=2000, random_state=42))])),
]
if lgbm_tuned is not None:
    stacking_estimators.append(('lgbm', clone(lgbm_tuned)))

# Meta-learner: scaled LR (inputs are probabilities, need no scaling, but
# passthrough=False means it only sees probs — all between 0 and 1 already)
meta_learner = LogisticRegression(
    C=1.0, max_iter=3000, class_weight='balanced',
    solver='lbfgs', random_state=42,
)

stack_cv = StratifiedKFold(
    n_splits=min(5, max(2, min(int((y_tr_aug==1).sum()),
                               int((y_tr_aug==0).sum())))),
    shuffle=True, random_state=42,
)

stacking_clf = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=meta_learner,
    cv=stack_cv,
    stack_method='predict_proba',
    passthrough=False,   # meta-learner sees only probs, not raw features
    n_jobs=-1,
)

try:
    stacking_clf.fit(X_tr_aug, y_tr_aug)
    model_configs['Stacking Ensemble'] = (stacking_clf, False)
    print('Stacking Ensemble ready')
except Exception as e:
    print(f'Stacking Ensemble failed: {e}')
    # Fallback: soft-voting
    from sklearn.ensemble import VotingClassifier
    from sklearn.calibration import CalibratedClassifierCV
    calib_members = [
        ('rf',  CalibratedClassifierCV(rf_tuned,  cv='prefit', method='isotonic')),
        ('et',  CalibratedClassifierCV(et_tuned,  cv='prefit', method='isotonic')),
        ('xgb', CalibratedClassifierCV(xgb_tuned, cv='prefit', method='isotonic')),
    ]
    if lgbm_tuned is not None:
        calib_members.append(
            ('lgbm', CalibratedClassifierCV(lgbm_tuned, cv='prefit', method='isotonic')))
    for _, c in calib_members: c.fit(X_tr_aug, y_tr_aug)
    vc = VotingClassifier(estimators=calib_members, voting='soft', n_jobs=-1)
    vc.estimators_ = [c for _, c in calib_members]
    vc.le_ = type('LE', (), {'classes_': np.array([0, 1])})()
    model_configs['Ensemble (Voting)'] = (vc, False)
    stacking_clf = vc
    print('Fallback Voting Ensemble ready')

# ── Cross-validation setup ────────────────────────────────────────────────────
cv_folds     = min(5, max(2, len(np.unique(groups[tr]))))
cv           = GroupKFold(n_splits=cv_folds)   # group-aware — no system leakage
groups_tr    = groups[tr]   # pass to cross_val_score

def find_optimal_threshold(y_true, y_probs):
    """Youden's J statistic: maximise (sensitivity + specificity - 1)."""
    try:
        fpr, tpr, thresholds = roc_curve(y_true, y_probs)
        j_idx = int(np.argmax(tpr - fpr))
        return min(float(thresholds[j_idx]), 0.5)
    except Exception:
        return 0.5

# ── Train, evaluate, record ───────────────────────────────────────────────────
all_model_metrics = {}
model_objects     = {}
model_thresholds  = {}   # per-model optimal decision threshold

header  = '  {:<25}  {:>8}  {:>9}  {:>8}  {:>8}  {:>8}  {:>7}  {:>7}  {:>8}  {:>9}/{:>5}  {:>6}'
divider = '  ' + '-' * 130
print('\n' + header.format('Model','Accuracy','Precision','Recall','F1',
                            'AUC','PR-AUC','BalAcc','MCC','CV F1','±σ','Thr'))
print(divider)

for name, (clf, _use_sc) in model_configs.items():
    # All models now receive raw (unscaled) X because tree models are
    # scale-invariant, and linear models have their own RobustScaler Pipeline.
    Xtr_  = X_tr_aug
    Xval_ = X_feat[val]

    try:
        if name == 'XGBoost':
            clf.fit(Xtr_, y_tr_aug,
                    eval_set=[(Xval_, y[val])], verbose=False)
        elif name not in ('Stacking Ensemble', 'Ensemble (Voting)'):
            clf.fit(Xtr_, y_tr_aug)
    except Exception as e:
        print(f'  {name} fit failed: {e}')
        continue

    preds = clf.predict(Xval_)
    try:    probs = clf.predict_proba(Xval_)[:, 1]
    except: probs = preds.astype(float)

    # Per-model optimal threshold
    thr = find_optimal_threshold(y[val], probs)
    model_thresholds[name] = thr
    preds_thr = (probs >= thr).astype(int)

    try:    auc    = float(roc_auc_score(y[val], probs))
    except: auc    = 0.5
    try:    pr_auc = float(average_precision_score(y[val], probs))
    except: pr_auc = float(precision_score(y[val], preds_thr, zero_division=0))
    bal_acc = float(balanced_accuracy_score(y[val], preds_thr))

    mcc = float(matthews_corrcoef(y[val], preds_thr))
    cm  = confusion_matrix(y[val], preds_thr, labels=[0, 1])
    tn, fp_c, fn_c, tp_c = (int(cm[0,0]), int(cm[0,1]),
                             int(cm[1,0]), int(cm[1,1])) if cm.shape==(2,2) else (0,0,0,0)

    try:
        cv_s = cross_val_score(clone(clf), X_tr_aug, y_tr_aug,
                               cv=cv, groups=groups_tr,
                               scoring='f1', error_score=0.0)
        cv_f1_mean = float(np.mean(cv_s)); cv_f1_std = float(np.std(cv_s))
    except Exception:
        cv_f1_mean = float(f1_score(y[val], preds_thr, zero_division=0))
        cv_f1_std  = 0.0

    m = {
        'accuracy':   float(accuracy_score(y[val], preds_thr)),
        'precision':  float(precision_score(y[val], preds_thr, zero_division=0)),
        'recall':     float(recall_score(y[val], preds_thr, zero_division=0)),
        'f1':         float(f1_score(y[val], preds_thr, zero_division=0)),
        'auc':        auc,
        'pr_auc':     pr_auc,
        'mcc':        mcc,
        'bal_acc':    bal_acc,
        'cv_f1_mean': cv_f1_mean,
        'cv_f1_std':  cv_f1_std,
        'tp': tp_c, 'fp': fp_c, 'fn': fn_c, 'tn': tn,
    }
    all_model_metrics[name] = m
    model_objects[name]     = (clf, probs, preds_thr, False)

    print('  {:<25}  {:>8.4f}  {:>9.4f}  {:>8.4f}  {:>8.4f}  {:>8.4f}'
          '  {:>7.4f}  {:>7.4f}  {:>8.3f}  {:>9.4f}/{:.4f}  {:>6.3f}'.format(
          name, m['accuracy'], m['precision'], m['recall'], m['f1'],
          auc, pr_auc, bal_acc, mcc, cv_f1_mean, cv_f1_std, thr))

# ── Final held-out TEST SET evaluation (§11) ────────────────────────────────
if len(test) > 0:
    print('\n══ FINAL TEST SET EVALUATION (held-out) ══')
    print(f'  n={len(test)} | Planets: {int(y[test].sum())} | FP: {int((y[test]==0).sum())}')
    test_results = {}
    for mname, (clf_obj, probs_val, preds_val, _) in model_objects.items():
        try:
            X_test_in = X_feat[test]
            probs_test = clf_obj.predict_proba(X_test_in)[:, 1]
            thr_test   = model_thresholds.get(mname, 0.5)
            preds_test = (probs_test >= thr_test).astype(int)
            try:    auc_t = float(roc_auc_score(y[test], probs_test))
            except: auc_t = 0.5
            try:    prauc_t = float(average_precision_score(y[test], probs_test))
            except: prauc_t = 0.0
            f1_t  = float(f1_score(y[test], preds_test, zero_division=0))
            mcc_t = float(matthews_corrcoef(y[test], preds_test))
            ba_t  = float(balanced_accuracy_score(y[test], preds_test))
            test_results[mname] = {'f1': f1_t, 'auc': auc_t, 'pr_auc': prauc_t,
                                    'mcc': mcc_t, 'bal_acc': ba_t}
            print(f'  {mname:<25}  F1={f1_t:.4f}  AUC={auc_t:.4f}  PR-AUC={prauc_t:.4f}'
                  f'  MCC={mcc_t:.3f}  BalAcc={ba_t:.4f}')
        except Exception as e:
            print(f'  {mname}: test eval failed — {e}')
    print('Test results stored in test_results dict.')
else:
    print('\n(No held-out test set — dataset too small for 3-way split)')
    test_results = {}

# ── Backward-compat aliases ───────────────────────────────────────────────────
xgb_model   = model_objects['XGBoost'][0]
xgb_metrics = all_model_metrics['XGBoost']
y_pred_xgb  = model_objects['XGBoost'][2]
y_prob_xgb  = model_objects['XGBoost'][1]

# ── SHAP on XGBoost ───────────────────────────────────────────────────────────
explainer = shap.TreeExplainer(xgb_model)
shap_vals = explainer.shap_values(X_feat)

plt.figure(figsize=(11, 7))
shap.summary_plot(shap_vals, X_feat, feature_names=feat_names, show=False, plot_size=None)
plt.title(f'SHAP Feature Importance — XGBoost ({len(feat_names)} features)', fontsize=12)
plt.tight_layout()
shap_summary_b64 = fig_to_b64(plt.gcf(), dpi=80)
plt.savefig(f'{RESULTS_DIR}/shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'SHAP plot saved  ({len(shap_summary_b64)//1024} KB base64)')

# ── Explainability metrics ────────────────────────────────────────────────────
mean_shap_abs = np.abs(shap_vals).mean(axis=0)
top3_idx      = np.argsort(-mean_shap_abs)[:3]
X_masked      = X_feat[val].copy()
for idx in top3_idx:
    X_masked[:, idx] = np.median(X_feat[:, idx])

acc_full     = float(accuracy_score(y[val], y_pred_xgb))
acc_masked   = float(accuracy_score(y[val], xgb_model.predict(X_masked)))
faithfulness = float(acc_full - acc_masked)
sparsity     = float((np.abs(shap_vals) < 0.01).mean())
if len(shap_vals) >= 3:
    corrs = [spearmanr(np.abs(shap_vals[a]), np.abs(shap_vals[b]))[0]
             for a in range(len(shap_vals))
             for b in range(a + 1, len(shap_vals))]
    consistency = float(np.nanmean([c for c in corrs if not np.isnan(c)])) if corrs else 1.0
else:
    consistency = 1.0

explainability_metrics = {
    'shap_faithfulness_drop': round(faithfulness, 4),
    'shap_sparsity':          round(sparsity, 4),
    'shap_consistency':       round(consistency, 4),
}
print(f'\nFaithfulness: {faithfulness:+.4f}  |  Sparsity: {sparsity:.4f}'
      f'  |  Consistency: {consistency:.4f}')
print(f'\nTop-3 SHAP features: {[feat_names[i] for i in top3_idx]}')

xgb_model.save_model(f'{MODELS_DIR}/xgboost.json')
print(f'XGBoost saved to {MODELS_DIR}/xgboost.json')


---
## 🧠 Cell 7 — 1D CNN + Grad-CAM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Local-view extractor ──────────────────────────────────────────────────────
LOCAL_LEN = 200   # zoomed-in view of transit region (10 % of global)

def extract_local_view(flux_array, center=0.5, width=0.14, out_len=LOCAL_LEN):
    """
    Extract a zoomed-in view centred on the transit (phase ≈ 0.5).
    The local view captures ingress/egress detail that the global view blurs.
    """
    N  = len(flux_array)
    lo = max(0, int((center - width / 2) * N))
    hi = min(N, int((center + width / 2) * N))
    seg = flux_array[lo:hi] if (hi - lo) >= 5 else flux_array
    return np.interp(
        np.linspace(0, 1, out_len), np.linspace(0, 1, len(seg)), seg
    ).astype(np.float32)

X_local  = np.array([extract_local_view(f) for f in processed])  # (N, 200)
X_global = X_raw                                                   # (N, 2000)

# ── Class weights ──────────────────────────────────────────────────────────────
n_total_cnn = len(y)
cnn_w = torch.tensor(
    [n_total_cnn / (2.0 * max((y == 0).sum(), 1)),
     n_total_cnn / (2.0 * max((y == 1).sum(), 1))],
    dtype=torch.float32,
).to(device)
print(f'CNN class weights: non-planet={cnn_w[0]:.3f}  planet={cnn_w[1]:.3f}')

# ── Building blocks ───────────────────────────────────────────────────────────

class ResBlock1D(nn.Module):
    """
    Residual block with skip connection.
    Helps gradient flow in deeper networks and reduces overfitting.
    """
    def __init__(self, channels, k=3, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, k, padding=k // 2, bias=False),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, k, padding=k // 2, bias=False),
            nn.BatchNorm1d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.net(x) + x)   # skip connection


class ViewEncoder(nn.Module):
    """
    Shared encoder architecture used for both the global (2000-pt) and
    local (200-pt) views.  Two strided conv blocks + two residual blocks +
    adaptive global-average pooling → fixed-size embedding.
    """
    def __init__(self, out_dim=128):
        super().__init__()
        # Strided conv blocks for rapid temporal downsampling
        self.layer1 = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(4),
        )
        self.res1   = ResBlock1D(32)
        self.layer2 = nn.Sequential(
            nn.Conv1d(32, 64, 5, padding=2, bias=False),
            nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(4),
        )
        self.res2   = ResBlock1D(64)
        # Final conv: 64 → 128 channels  (Grad-CAM hooks HERE for global enc)
        self.layer3 = nn.Sequential(
            nn.Conv1d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm1d(128), nn.GELU(),
        )
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, out_dim),
            nn.GELU(),
            nn.Dropout(0.35),
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.res1(x)
        x = self.layer2(x)
        x = self.res2(x)
        x = self.layer3(x)     # ← Grad-CAM forward hook target
        return self.proj(self.gap(x))


class DualViewCNN(nn.Module):
    """
    Expert dual-view architecture (inspired by AstroNet, Shallue & Vanderburg 2018).

    • Global branch: 2000-point full phase-folded LC — captures periodic context.
    • Local branch:  200-point zoomed view around transit — captures shape detail
                     (flat bottom vs V-shape, ingress/egress slope).
    • Channel attention gate: lets the network weight which embedding dims matter.
    • Classification head: 192 → 64 → 2 logits.
    """
    def __init__(self, g_dim=128, l_dim=64):
        super().__init__()
        self.global_enc = ViewEncoder(out_dim=g_dim)
        self.local_enc  = ViewEncoder(out_dim=l_dim)
        total = g_dim + l_dim
        # Channel attention: squeeze-and-excitation style gate
        self.attn = nn.Sequential(
            nn.Linear(total, total // 4), nn.GELU(),
            nn.Linear(total // 4, total), nn.Sigmoid(),
        )
        self.head = nn.Sequential(
            nn.Linear(total, 64), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(64, 2),
        )

    def forward(self, x_global, x_local):
        g   = self.global_enc(x_global)
        l   = self.local_enc(x_local)
        cat = torch.cat([g, l], dim=1)
        return self.head(cat * self.attn(cat))   # gated combination


# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ls    = label_smoothing

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.alpha,
                              reduction='none', label_smoothing=self.ls)
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


# ── Augmentation ──────────────────────────────────────────────────────────────
def augment_dual(xb_g, xb_l):
    """Gaussian noise + random circular shift on global view only."""
    xb_g = xb_g + torch.randn_like(xb_g) * 0.003
    xb_l = xb_l + torch.randn_like(xb_l) * 0.004
    shift = int(np.random.randint(-int(SEQ_LEN * 0.04), int(SEQ_LEN * 0.04) + 1))
    if shift != 0:
        xb_g = torch.roll(xb_g, shifts=shift, dims=-1)
    return xb_g, xb_l


def mixup_batch(xb_g, xb_l, yb, alpha=0.15):
    """
    Mixup augmentation: interpolate between training examples.
    Creates virtual training examples that lie 'between' real ones,
    reducing overfitting and improving calibration on small datasets.
    """
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(xb_g.size(0), device=xb_g.device)
    mg  = lam * xb_g + (1 - lam) * xb_g[idx]
    ml  = lam * xb_l + (1 - lam) * xb_l[idx]
    return mg, ml, yb, yb[idx], lam


# ── Data preparation ──────────────────────────────────────────────────────────
Xg_tr = torch.tensor(X_global[tr][:, None, :], dtype=torch.float32)
Xl_tr = torch.tensor(X_local[tr][:, None, :],  dtype=torch.float32)
yt    = torch.tensor(y[tr], dtype=torch.long)
Xg_v  = torch.tensor(X_global[val][:, None, :], dtype=torch.float32)
Xl_v  = torch.tensor(X_local[val][:, None, :],  dtype=torch.float32)
yv    = y[val]

loader = DataLoader(TensorDataset(Xg_tr, Xl_tr, yt),
                    batch_size=CNN_BATCH_SIZE, shuffle=True, drop_last=False)

# ── Model, optimiser, scheduler ───────────────────────────────────────────────
# Reduce capacity for very small datasets to avoid overfitting
g_dim = 64  if len(tr) < 30 else 128
l_dim = 32  if len(tr) < 30 else 64
cnn       = DualViewCNN(g_dim=g_dim, l_dim=l_dim).to(device)
optimizer = torch.optim.AdamW(cnn.parameters(), lr=CNN_LR, weight_decay=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1, eta_min=5e-6)
criterion = FocalLoss(alpha=cnn_w, gamma=2.0)

n_params = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
print(f'DualViewCNN  trainable params: {n_params:,}')

# ── Training loop ─────────────────────────────────────────────────────────────
best_f1, patience_cnt = 0.0, 0
best_state = {k: v.clone() for k, v in cnn.state_dict().items()}  # init prevents None crash if F1 never improves
history   = []
USE_MIXUP = True

for epoch in range(CNN_EPOCHS):
    cnn.train()
    epoch_loss = 0.0

    for xb_g, xb_l, yb in loader:
        xb_g, xb_l, yb = xb_g.to(device), xb_l.to(device), yb.to(device)

        # Augment
        xb_g, xb_l = augment_dual(xb_g, xb_l)

        # Mixup (50 % probability per batch)
        if USE_MIXUP and np.random.rand() > 0.5 and len(yb) > 1:
            xb_g, xb_l, ya, yb_, lam = mixup_batch(xb_g, xb_l, yb)
            optimizer.zero_grad()
            logits = cnn(xb_g, xb_l)
            loss   = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_)
        else:
            optimizer.zero_grad()
            logits = cnn(xb_g, xb_l)
            loss   = criterion(logits, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(cnn.parameters(), 2.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()

    # ── Validation ────────────────────────────────────────────────────────
    cnn.eval()
    with torch.no_grad():
        val_logits = cnn(Xg_v.to(device), Xl_v.to(device))
        val_loss   = F.cross_entropy(val_logits,
                                     torch.tensor(yv, dtype=torch.long).to(device)).item()
        val_preds  = val_logits.argmax(1).cpu().numpy()

    f1  = f1_score(yv, val_preds, zero_division=0)
    acc = accuracy_score(yv, val_preds)
    history.append({'epoch': epoch + 1,
                    'train_loss': round(epoch_loss / max(len(loader), 1), 5),
                    'val_loss':   round(val_loss, 5),
                    'f1':  round(f1,  4),
                    'acc': round(acc, 4)})

    if f1 >= best_f1:
        best_f1    = f1
        best_state = {k: v.clone() for k, v in cnn.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= 18:
            print(f'  Early stop at epoch {epoch + 1}')
            break

    if (epoch + 1) % 10 == 0:
        print(f'  Ep {epoch+1:3d}/{CNN_EPOCHS}'
              f'  train_loss={epoch_loss/max(len(loader),1):.4f}'
              f'  val_loss={val_loss:.4f}  F1={f1:.4f}  Acc={acc:.4f}')

cnn.load_state_dict(best_state)
cnn.eval()

# ── Val-set metrics ───────────────────────────────────────────────────────────
with torch.no_grad():
    val_logits_f  = cnn(Xg_v.to(device), Xl_v.to(device))
    y_prob_cnn    = torch.softmax(val_logits_f, dim=1)[:, 1].cpu().numpy()
    y_pred_cnn_05 = val_logits_f.argmax(1).cpu().numpy()

try:
    if len(np.unique(yv)) < 2:
        print('Warning: val set has only one class — AUC set to 0.5')
        cnn_auc = 0.5
    else:
        cnn_auc = float(roc_auc_score(yv, y_prob_cnn))
except: cnn_auc = 0.5

# Youden's J optimal threshold for CNN
from sklearn.metrics import roc_curve as _roc_curve
try:
    _fpr, _tpr, _thr = _roc_curve(yv, y_prob_cnn)
    cnn_threshold = min(float(_thr[int(np.argmax(_tpr - _fpr))]), 0.5)
except Exception:
    cnn_threshold = 0.5
print(f'CNN Youden threshold: {cnn_threshold:.3f}')

y_pred_cnn = (y_prob_cnn >= cnn_threshold).astype(int)

from sklearn.metrics import matthews_corrcoef
cnn_metrics = {
    'accuracy':  float(accuracy_score(yv, y_pred_cnn)),
    'precision': float(precision_score(yv, y_pred_cnn, zero_division=0)),
    'recall':    float(recall_score(yv, y_pred_cnn, zero_division=0)),
    'f1':        float(f1_score(yv, y_pred_cnn, zero_division=0)),
    'auc':       cnn_auc,
    'mcc':       float(matthews_corrcoef(yv, y_pred_cnn)),
}
print(f'Best val F1: {best_f1:.4f}')
print('CNN metrics:', {k: round(v, 4) for k, v in cnn_metrics.items()})

# ── ALL-STAR inference with Test-Time Augmentation (TTA) ────────────────────
#
# Average predictions over 7 augmented passes to reduce variance.
# Each pass adds small independent noise; the average is smoother and
# better-calibrated than a single forward pass.
#
print('\nRunning Test-Time Augmentation (7 passes)...')
Xg_all = torch.tensor(X_global[:, None, :], dtype=torch.float32)
Xl_all = torch.tensor(X_local[:, None, :],  dtype=torch.float32)

tta_probs = []
for _ in range(7):
    aug_g = Xg_all + torch.randn_like(Xg_all) * 0.001
    aug_l = Xl_all + torch.randn_like(Xl_all) * 0.001
    with torch.no_grad():
        logits = cnn(aug_g.to(device), aug_l.to(device))
        p      = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    tta_probs.append(p)

all_probs_cnn = np.mean(tta_probs, axis=0)                          # TTA mean
all_preds_cnn = (all_probs_cnn >= cnn_threshold).astype(int)
print(f'TTA complete — predicted {all_preds_cnn.sum()} planets / {len(all_preds_cnn)} stars')

# ── Training-curve figure → base64 ───────────────────────────────────────────
fig_tr, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
fig_tr.patch.set_facecolor('#0b0f1a')
for ax in [ax1, ax2]:
    ax.set_facecolor('#111827'); ax.tick_params(colors='gray')
    ax.spines[:].set_color('#1e2d47')
eps = [h['epoch'] for h in history]
ax1.plot(eps, [h['train_loss'] for h in history], color='#3b82f6', lw=1.5, label='Train')
ax1.plot(eps, [h['val_loss']   for h in history], color='#f59e0b', lw=1.5, label='Val')
ax1.set_title('Focal-Loss Curves', color='white')
ax1.set_xlabel('Epoch', color='gray'); ax1.set_ylabel('Loss', color='gray')
ax1.legend(facecolor='#1a2235', labelcolor='white')
ax2.plot(eps, [h['f1']  for h in history], color='#10b981', lw=1.5, label='Val F1')
ax2.plot(eps, [h['acc'] for h in history], color='#8b5cf6', lw=1.5, label='Val Acc')
ax2.set_title('F1 & Accuracy', color='white')
ax2.set_xlabel('Epoch', color='gray')
ax2.legend(facecolor='#1a2235', labelcolor='white')
plt.tight_layout()
training_curve_b64 = fig_to_b64(fig_tr, dpi=90)
plt.savefig(f'{RESULTS_DIR}/training_curve.png', dpi=100, bbox_inches='tight',
            facecolor='#0b0f1a')
plt.show()
print(f'Training curve saved ({len(training_curve_b64)//1024} KB base64)')


--- ## 🔬 Cell 7b — CNN Ablation Study

Compare: Global-only CNN, Local-only CNN, Global+Local (no attention), Full DualViewCNN.
This validates why each architectural component contributes to the final performance.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ABLATION STUDY — DualViewCNN component contributions                         
# Compares: Global-only | Local-only | Global+Local (no gate) | Full model      
# ══════════════════════════════════════════════════════════════════════════════
print("\n══ CNN Ablation Study ══")
from sklearn.metrics import f1_score as _f1, roc_auc_score as _auc, balanced_accuracy_score as _balacc

# Build datasets (reuse from Cell 7)
Xg_all = torch.tensor(X_global[:, None, :], dtype=torch.float32)
Xl_all = torch.tensor(X_local[:, None, :],  dtype=torch.float32)
y_all  = torch.tensor(y, dtype=torch.long)
yv     = y[val]

def _eval_model(model, device):
    model.eval()
    with torch.no_grad():
        logits = model(Xg_all[val].to(device), Xl_all[val].to(device))
        probs  = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
    preds = (probs >= 0.5).astype(int)
    try:    auc = float(_auc(yv, probs))
    except: auc = 0.5
    return {
        'F1':       round(float(_f1(yv, preds, zero_division=0)), 4),
        'AUC-ROC':  round(auc, 4),
        'Bal.Acc':  round(float(_balacc(yv, preds)), 4),
    }

class _GlobalOnlyCNN(nn.Module):
    """Uses only the global branch; local replaced by zeros."""
    def __init__(self, base):
        super().__init__()
        self.global_enc = base.global_enc
        self.head = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.4), nn.Linear(64, 2))
    def forward(self, xg, xl):
        return self.head(self.global_enc(xg))

class _LocalOnlyCNN(nn.Module):
    """Uses only the local branch."""
    def __init__(self, base):
        super().__init__()
        self.local_enc = base.local_enc
        self.head = nn.Sequential(nn.Linear(64, 32), nn.GELU(), nn.Dropout(0.4), nn.Linear(32, 2))
    def forward(self, xg, xl):
        return self.head(self.local_enc(xl))

class _NoGateCNN(nn.Module):
    """Global+Local concatenated without channel-attention gate."""
    def __init__(self, base):
        super().__init__()
        self.global_enc = base.global_enc
        self.local_enc  = base.local_enc
        total = 128 + 64
        self.head = nn.Sequential(nn.Linear(total, 64), nn.GELU(), nn.Dropout(0.4), nn.Linear(64, 2))
    def forward(self, xg, xl):
        return self.head(torch.cat([self.global_enc(xg), self.local_enc(xl)], dim=1))

ablation_variants = {
    'Global-only CNN':       _GlobalOnlyCNN(cnn),
    'Local-only CNN':        _LocalOnlyCNN(cnn),
    'Global+Local (no gate)':_NoGateCNN(cnn),
    'Full DualViewCNN':      cnn,
}

print(f"  {'Variant':<30}  {'F1':>8}  {'AUC-ROC':>9}  {'Bal.Acc':>8}")
print("  " + "-"*60)
ablation_results = {}
for vname, vmodel in ablation_variants.items():
    vmodel.to(device)
    res = _eval_model(vmodel, device)
    ablation_results[vname] = res
    print(f"  {vname:<30}  {res['F1']:>8.4f}  {res['AUC-ROC']:>9.4f}  {res['Bal.Acc']:>8.4f}")

print("\nAblation study complete — stored in ablation_results")


---
## 🧠 Cell 7c — BiLSTM + Attention (recurrent deep branch)

Upgrades the deep-learning stack with a bidirectional LSTM. The BiLSTM reads the entire
2000-point phase-folded light curve in both directions, and a soft-attention head scores
each time step. Attention weights double as an explainability map (“recurrent Grad-CAM”)
without needing external gradients.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ── BiLSTMNet: bidirectional LSTM + soft attention over time steps ──────────────────────────────────────────────
class BiLSTMNet(nn.Module):
    """
    Bidirectional LSTM for light curve classification.

    • Reads the phase-folded flux in both directions (forward + backward context).
    • Soft-attention head averages hidden states, emphasising the transit region.
    • Attention weights are exported as an explainability map (like Grad-CAM).
    """
    def __init__(self, hidden=64, layers=2, dropout=0.3, num_classes=2):
        super().__init__()
        self.rnn = nn.LSTM(input_size=1, hidden_size=hidden,
                           num_layers=layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0.0,
                           bidirectional=True)
        self.attn = nn.Sequential(
            nn.Linear(2 * hidden, 64), nn.Tanh(), nn.Linear(64, 1),
        )
        self.head = nn.Sequential(
            nn.Linear(2 * hidden, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        out, _ = self.rnn(x.unsqueeze(-1))                  # (N, T, 2*hidden)
        w = torch.softmax(self.attn(out).squeeze(-1), dim=1)  # (N, T)
        ctx = (out * w.unsqueeze(-1)).sum(dim=1)            # (N, 2*hidden)
        return self.head(ctx)


def compute_bilstm_attention(model, flux_array):
    """
    Return the soft-attention weights over the light curve, normalised to [0, 1].
    This is the recurrent analogue of Grad-CAM: it shows which phase regions
    the BiLSTM relies on for its decision.
    """
    model.eval()
    x = torch.tensor(flux_array[None, :], dtype=torch.float32).to(device)
    with torch.no_grad():
        out, _ = model.rnn(x.unsqueeze(-1))
        attn = torch.softmax(model.attn(out).squeeze(-1), dim=1)[0].cpu().numpy()
    att = np.interp(np.linspace(0, 1, len(flux_array)),
                    np.linspace(0, 1, len(attn)), attn)
    return att / att.max() if att.max() > 0 else att


# ── Model ─────────────────────────────────────────────────────────────────────────────────────────
bilstm   = BiLSTMNet().to(device)
bilstm_w = cnn_w  # reuse the same class weights (same class imbalance)

opt_b  = torch.optim.AdamW(bilstm.parameters(), lr=CNN_LR, weight_decay=3e-4)
sch_b  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_b, T_0=20, T_mult=1, eta_min=5e-6)
crit_b = FocalLoss(alpha=bilstm_w, gamma=2.0)

Xb_tr = torch.tensor(X_global[tr], dtype=torch.float32)
Xb_v  = torch.tensor(X_global[val], dtype=torch.float32)
yb_tr = torch.tensor(y[tr], dtype=torch.long)
yb_v  = y[val]
loader_b = DataLoader(TensorDataset(Xb_tr, yb_tr),
                      batch_size=CNN_BATCH_SIZE, shuffle=True, drop_last=False)

print(f'BiLSTM  trainable params: {sum(p.numel() for p in bilstm.parameters() if p.requires_grad):,}')

# ── Training loop (same schedule / early-stop as the CNN) ─────────────────────────────────────────────────
best_f1_b, patience_b = 0.0, 0
best_state_b = {k: v.clone() for k, v in bilstm.state_dict().items()}
hist_b = []

for epoch in range(CNN_EPOCHS):
    bilstm.train()
    epoch_loss = 0.0
    for xb, yb in loader_b:
        xb = xb + torch.randn_like(xb) * 0.002          # noise augmentation
        xb, yb = xb.to(device), yb.to(device)
        opt_b.zero_grad()
        loss = crit_b(bilstm(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bilstm.parameters(), 2.0)
        opt_b.step()
        epoch_loss += loss.item()
    sch_b.step()

    bilstm.eval()
    with torch.no_grad():
        vlogits = bilstm(Xb_v.to(device))
        vpreds  = vlogits.argmax(1).cpu().numpy()
    f1_b  = f1_score(yb_v, vpreds, zero_division=0)
    acc_b = accuracy_score(yb_v, vpreds)
    hist_b.append({'epoch': epoch + 1,
                   'train_loss': round(epoch_loss / max(len(loader_b), 1), 5),
                   'f1': round(f1_b, 4), 'acc': round(acc_b, 4)})

    if f1_b >= best_f1_b:
        best_f1_b   = f1_b
        best_state_b = {k: v.clone() for k, v in bilstm.state_dict().items()}
        patience_b  = 0
    else:
        patience_b += 1
        if patience_b >= 18:
            print(f'  Early stop at epoch {epoch + 1}')
            break

    if (epoch + 1) % 10 == 0:
        print(f'  Ep {epoch+1:3d}/{CNN_EPOCHS}'
              f'  train_loss={epoch_loss/max(len(loader_b),1):.4f}'
              f'  F1={f1_b:.4f}  Acc={acc_b:.4f}')

bilstm.load_state_dict(best_state_b)
bilstm.eval()

# ── Val-set metrics ─────────────────────────────────────────────────────────────────────────────────────────
with torch.no_grad():
    vprobs_b = torch.softmax(bilstm(Xb_v.to(device)), dim=1)[:, 1].cpu().numpy()
try:
    if len(np.unique(yb_v)) < 2:
        bilstm_auc = 0.5
    else:
        bilstm_auc = float(roc_auc_score(yb_v, vprobs_b))
except: bilstm_auc = 0.5

from sklearn.metrics import roc_curve as _roc_curve_b
try:
    _fpr_b, _tpr_b, _thr_b = _roc_curve_b(yb_v, vprobs_b)
    bilstm_threshold = min(float(_thr_b[int(np.argmax(_tpr_b - _fpr_b))]), 0.5)
except Exception:
    bilstm_threshold = 0.5
print(f'BiLSTM Youden threshold: {bilstm_threshold:.3f}')

vpreds_b = (vprobs_b >= bilstm_threshold).astype(int)
bilstm_metrics = {
    'accuracy':  float(accuracy_score(yb_v, vpreds_b)),
    'precision': float(precision_score(yb_v, vpreds_b, zero_division=0)),
    'recall':    float(recall_score(yb_v, vpreds_b, zero_division=0)),
    'f1':        float(f1_score(yb_v, vpreds_b, zero_division=0)),
    'auc':       bilstm_auc,
    'mcc':       float(matthews_corrcoef(yb_v, vpreds_b)),
}
print(f'Best val F1: {best_f1_b:.4f}')
print('BiLSTM metrics:', {k: round(v, 4) for k, v in bilstm_metrics.items()})

# ── ALL-star inference with Test-Time Augmentation (7 passes) ─────────────────────────────────────────
print('Running BiLSTM Test-Time Augmentation (7 passes)...')
Xb_all = torch.tensor(X_global, dtype=torch.float32)
tta_probs_b = []
for _ in range(7):
    aug_b = Xb_all + torch.randn_like(Xb_all) * 0.001
    with torch.no_grad():
        p = torch.softmax(bilstm(aug_b.to(device)), dim=1)[:, 1].cpu().numpy()
    tta_probs_b.append(p)

all_probs_bilstm = np.mean(tta_probs_b, axis=0)
all_preds_bilstm = (all_probs_bilstm >= bilstm_threshold).astype(int)
print(f'TTA complete — BiLSTM predicted {all_preds_bilstm.sum()} planets / {len(all_preds_bilstm)} stars')

# ── Attention training figure ─────────────────────────────────────────────────────────────────────────────────────────
fig_bt, (ax1b, ax2b) = plt.subplots(1, 2, figsize=(11, 3.5))
fig_bt.patch.set_facecolor('#0b0f1a')
for ax in [ax1b, ax2b]:
    ax.set_facecolor('#111827'); ax.tick_params(colors='gray')
    ax.spines[:].set_color('#1e2d47')
epsb = [h['epoch'] for h in hist_b]
ax1b.plot(epsb, [h['train_loss'] for h in hist_b], color='#3b82f6', lw=1.5, label='Train')
ax1b.set_title('BiLSTM Loss', color='white'); ax1b.set_xlabel('Epoch', color='gray')
ax1b.legend(facecolor='#1a2235', labelcolor='white')
ax2b.plot(epsb, [h['f1']  for h in hist_b], color='#10b981', lw=1.5, label='Val F1')
ax2b.plot(epsb, [h['acc'] for h in hist_b], color='#8b5cf6', lw=1.5, label='Val Acc')
ax2b.set_title('BiLSTM F1 & Accuracy', color='white'); ax2b.set_xlabel('Epoch', color='gray')
ax2b.legend(facecolor='#1a2235', labelcolor='white')
plt.tight_layout()
bilstm_training_b64 = fig_to_b64(fig_bt, dpi=90)
plt.savefig(f'{RESULTS_DIR}/bilstm_training.png', dpi=100, bbox_inches='tight',
            facecolor='#0b0f1a')
plt.show()
print(f'BiLSTM training curve saved ({len(bilstm_training_b64)//1024} KB base64)')

---
## 🌡️ Cell 8 — Grad-CAM + Export JSON for each star

In [ ]:
import math

def compute_gradcam(model, flux_array, local_array, target_class=1):
    """
    Grad-CAM on the DualViewCNN global encoder's last conv layer.
    Highlights which phase regions most influenced the classification.
    """
    model.eval()
    x_g = torch.tensor(flux_array[None, None, :],  dtype=torch.float32,
                        requires_grad=True).to(device)
    x_l = torch.tensor(local_array[None, None, :], dtype=torch.float32).to(device)
    fmaps, grads = {}, {}

    # Hook the last Conv1d in global_enc.layer3 (64→128 channels)
    h1 = model.global_enc.layer3[0].register_forward_hook(
             lambda m, i, o: fmaps.update({'x': o}))
    h2 = model.global_enc.layer3[0].register_backward_hook(
             lambda m, gi, go: grads.update({'x': go[0]}))

    logits = model(x_g, x_l)
    model.zero_grad()
    logits[0, target_class].backward()
    h1.remove(); h2.remove()

    g   = grads['x'].detach().cpu().numpy()[0]   # (128, T)
    f   = fmaps['x'].detach().cpu().numpy()[0]   # (128, T)
    cam = np.maximum((g.mean(axis=1)[:, None] * f).sum(axis=0), 0)
    cam = np.interp(np.linspace(0, 1, len(flux_array)),
                    np.linspace(0, 1, len(cam)), cam)
    return cam / cam.max() if cam.max() > 0 else cam


def gradcam_localization(cam, window=(0.45, 0.55)):
    N    = len(cam)
    lo, hi = int(window[0] * N), int(window[1] * N)
    total  = cam.sum()
    return float(cam[lo:hi].sum() / total) if total > 0 else 0.0

def _san(obj):
    if isinstance(obj, float):
        return None if (math.isnan(obj) or math.isinf(obj)) else obj
    if isinstance(obj, dict):  return {k: _san(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_san(v) for v in obj]
    return obj

def safe_metrics(m):
    if not m:
        return {'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0,
                'auc': 0.5, 'pr_auc': 0, 'mcc': 0, 'bal_acc': 0,
                'cv_f1_mean': 0, 'cv_f1_std': 0,
                'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}
    return {
        'accuracy':   float(m.get('accuracy',   0) or 0),
        'precision':  float(m.get('precision',  0) or 0),
        'recall':     float(m.get('recall',     0) or 0),
        'f1':         float(m.get('f1',         0) or 0),
        'auc':        float(m.get('auc',       0.5) or 0.5),
        'pr_auc':     float(m.get('pr_auc',     0) or 0),
        'mcc':        float(m.get('mcc',        0) or 0),
        'bal_acc':    float(m.get('bal_acc',    0) or 0),
        'cv_f1_mean': float(m.get('cv_f1_mean', m.get('f1', 0)) or 0),
        'cv_f1_std':  float(m.get('cv_f1_std',  0) or 0),
        'tp':  int(m.get('tp',  0) or 0),
        'fp':  int(m.get('fp',  0) or 0),
        'fn':  int(m.get('fn',  0) or 0),
        'tn':  int(m.get('tn',  0) or 0),
    }

results = []
phase   = np.linspace(0, 1, SEQ_LEN)

for i, sid in enumerate(all_ids):
    flux_proc = processed[i]

    # ── CNN prediction for THIS star (all-star inference, no 0.5 default) ──
    cnn_prob  = float(all_probs_cnn[i])
    pred_class = int(all_preds_cnn[i])   # 1 = planet, 0 = non-planet

    # ── BiLSTM prediction + attention for THIS star ─────────────────────────────────────────
    bilstm_prob  = float(all_probs_bilstm[i])
    bilstm_pred  = int(all_preds_bilstm[i])
    bilstm_att   = compute_bilstm_attention(bilstm, flux_proc)
    bl_focus     = gradcam_localization(bilstm_att)

    # ── Grad-CAM (always compute for predicted class to highlight correctly) ──
    gc_target = pred_class if pred_class in [0, 1] else 1
    gradcam   = compute_gradcam(cnn, flux_proc, X_local[i], target_class=gc_target)
    gc_loc    = gradcam_localization(gradcam)

    # ── SHAP for this star ────────────────────────────────────────────────────
    _sv_raw = explainer.shap_values(X_feat[i:i + 1])
    sv = (_sv_raw[1][0] if isinstance(_sv_raw, list) and len(_sv_raw) > 1
          else _sv_raw[0][0] if isinstance(_sv_raw, list)
          else _sv_raw[0])
    shap_top8 = sorted(
        [{'feature': n, 'val': float(v), 'pos': bool(float(v) >= 0)}
         for n, v in zip(feat_names, sv)],
        key=lambda x: -abs(x['val'])
    )[:8]

    # ── Per-model predictions for this star ───────────────────────────────────
    model_preds = {}
    for mname, (clf, _, _, use_sc) in model_objects.items():
        try:
            x_in = (X_val_sc if use_sc else X_feat)[i:i + 1] if False else \
                   (scaler.transform(X_feat[i:i + 1]) if use_sc else X_feat[i:i + 1])
            p = float(clf.predict_proba(x_in)[0, 1])
        except Exception:
            p = 0.5
        model_preds[mname] = {'prob': round(p, 4),
                              'pred': 'planet' if p >= model_thresholds.get(mname, 0.5) else 'false'}
    model_preds['BiLSTM'] = {'prob': round(bilstm_prob, 4),
                             'pred': 'planet' if bilstm_pred == 1 else 'false'}

    # ── Light curve image (per-star) ──────────────────────────────────────────
    fig_lc, ax_lc = plt.subplots(figsize=(8, 2.8))
    fig_lc.patch.set_facecolor('#0b0f1a')
    ax_lc.set_facecolor('#111827'); ax_lc.tick_params(colors='gray')
    ax_lc.spines[:].set_color('#1e2d47')
    ax_lc.plot(phase, flux_proc, color='#3b82f6', lw=0.9)
    gc_max = max(gradcam.max(), 1e-6)
    for j in range(len(phase) - 1):
        if gradcam[j] > 0.06:
            ax_lc.axvspan(phase[j], phase[j + 1],
                          alpha=float(gradcam[j] / gc_max) * 0.4,
                          color='#ef4444', linewidth=0)
    true_lbl = 'Planet' if y[i] == 1 else 'Non-planet'
    pred_lbl = 'Planet' if pred_class == 1 else 'Non-planet'
    flag = '  ⚠ MISMATCH' if true_lbl != pred_lbl else ''
    ax_lc.axvspan(0.45, 0.55, alpha=0.10, color='#10b981')
    ax_lc.set_title(f'{sid}  |  Archive: {true_lbl}  CNN: {pred_lbl}{flag}  '
                    f'conf={cnn_prob:.2f}  cam={gc_loc:.2f}',
                    color='white', fontsize=9)
    ax_lc.set_xlabel('Phase', color='gray', fontsize=8)
    ax_lc.set_ylabel('Norm. Flux', color='gray', fontsize=8)
    plt.tight_layout()
    img_flux = fig_to_b64(fig_lc, dpi=90)
    plt.savefig(f"{RESULTS_DIR}/{sid.replace('/','_')}_lc.png",
                dpi=100, bbox_inches='tight', facecolor='#0b0f1a')
    plt.show()
    plt.close(fig_lc)

    # ── Grad-CAM image (per-star) ─────────────────────────────────────────────
    fig_gc, (ax_gc1, ax_gc2) = plt.subplots(2, 1, figsize=(10, 4.5),
                                             sharex=True,
                                             gridspec_kw={'height_ratios': [3, 1]})
    fig_gc.patch.set_facecolor('#0b0f1a')
    for ax in [ax_gc1, ax_gc2]:
        ax.set_facecolor('#111827'); ax.tick_params(colors='gray')
        ax.spines[:].set_color('#1e2d47')
    ax_gc1.plot(phase, flux_proc, color='#3b82f6', lw=0.9)
    for j in range(len(phase) - 1):
        if gradcam[j] > 0.06:
            ax_gc1.axvspan(phase[j], phase[j + 1],
                           alpha=float(gradcam[j] / gc_max) * 0.45,
                           color='#ef4444', linewidth=0)
    ax_gc1.axvspan(0.45, 0.55, alpha=0.12, color='#10b981', label='Transit window')
    ax_gc1.set_ylabel('Norm. Flux', color='gray')
    ax_gc1.set_title(f'Grad-CAM  {sid}  |  Archive: {true_lbl}  CNN: {pred_lbl}{flag}  '
                     f'loc={gc_loc:.2f}', color='white', fontsize=9)
    ax_gc1.legend(facecolor='#1a2235', labelcolor='white', fontsize=8)
    ax_gc2.fill_between(phase, gradcam / gc_max, color='#ef4444', alpha=0.75)
    ax_gc2.axvspan(0.45, 0.55, alpha=0.15, color='#10b981')
    ax_gc2.set_xlabel('Phase', color='gray'); ax_gc2.set_ylabel('Activation', color='gray')
    plt.tight_layout()
    img_gradcam = fig_to_b64(fig_gc, dpi=90)
    plt.savefig(f"{RESULTS_DIR}/{sid.replace('/','_')}_gradcam.png",
                dpi=100, bbox_inches='tight', facecolor='#0b0f1a')
    plt.close(fig_gc)


    # -- Star & planet parameters from NASA archive ----------------------
    star_params = {}
    try:
        if all_missions[i] == 'Kepler' and koi_df is not None and sid in koi_df.index:
            row = koi_df.loc[sid]
            def _fv(col):
                v = row.get(col)
                try: return float(v) if v is not None and str(v).strip() not in ('', 'nan') else None
                except: return None
            star_params = {
                'kepid': str(int(row.get('kepid', 0) or 0)),
                'mission': 'Kepler',
                'period':  _fv('koi_period'),
                'duration':_fv('koi_duration'),
                'depth':   _fv('koi_depth'),
                'prad':    _fv('koi_prad'),
                'sma':     _fv('koi_sma'),
                'teq':     _fv('koi_teq'),
                'insol':   _fv('koi_insol'),
                'impact':  _fv('koi_impact'),
                'srad':    _fv('koi_srad'),
                'smass':   _fv('koi_smass'),
                'steff':   _fv('koi_steff'),
                'slogg':   _fv('koi_slogg'),
                'mag':     _fv('koi_kepmag'),
                'score':   _fv('koi_score'),
                'ra':      _fv('ra'),
                'dec':     _fv('dec'),
                'disposition': str(row.get('koi_pdisposition', '') or ''),
            }
        elif all_missions[i] == 'TESS' and toi_df is not None:
            tic_num = sid.replace('TIC-', '')
            if tic_num in toi_df.index:
                row = toi_df.loc[tic_num]
                def _fv(col):
                    v = row.get(col)
                    try: return float(v) if v is not None and str(v).strip() not in ('', 'nan') else None
                    except: return None
                star_params = {
                    'mission': 'TESS',
                    'period':  _fv('pl_orbper'),
                    'duration':_fv('pl_trandurh'),
                    'depth':   _fv('pl_trandep'),
                    'prad':    _fv('pl_rade'),
                    'sma':     _fv('pl_orbsmax'),
                    'teq':     _fv('pl_eqt'),
                    'insol':   _fv('pl_insol'),
                    'impact':  _fv('pl_imppar'),
                    'srad':    _fv('st_rad'),
                    'smass':   _fv('st_mass'),
                    'steff':   _fv('st_teff'),
                    'slogg':   _fv('st_logg'),
                    'mag':     _fv('st_tmag'),
                    'ra':      _fv('ra'),
                    'dec':     _fv('dec'),
                    'disposition': str(row.get('tfopwg_disp', '') or ''),
                }
    except Exception:
        pass

    # ── Assemble result dict ──────────────────────────────────────────────────
    gc400   = np.interp(np.linspace(0, 1, 400),
                        np.linspace(0, 1, len(gradcam)), gradcam).tolist()
    flux400 = np.interp(np.linspace(0, 1, 400),
                        np.linspace(0, 1, len(flux_proc)), flux_proc).tolist()

    result = {
        'id':         sid,
        'mission':    all_missions[i],
        'label':      'planet' if y[i] == 1 else 'false_positive',
        'prediction': 'planet' if pred_class == 1 else 'false',
        'xgb':        safe_metrics(all_model_metrics.get('XGBoost')),
        'cnn':        safe_metrics(cnn_metrics),
        'bilstm':     safe_metrics(bilstm_metrics),
        'models':     {k: safe_metrics(v) for k, v in all_model_metrics.items()},
        'model_preds': model_preds,
        'shap':       shap_top8,
        'explainability': {
            'shap_faithfulness_drop': float(explainability_metrics.get('shap_faithfulness_drop', 0) or 0),
            'shap_sparsity':          float(explainability_metrics.get('shap_sparsity',          0) or 0),
            'shap_consistency':       float(explainability_metrics.get('shap_consistency',        1) or 1),
            'gradcam_localization':   round(gc_loc, 4),
            'bilstm_localization':    round(bl_focus, 4),
        },
        'gradcam400':  [round(float(v), 4) if v is not None else 0.0 for v in gc400],
        'bilstmAttention400': [round(float(v), 4) for v in np.interp(
            np.linspace(0, 1, 400), np.linspace(0, 1, len(bilstm_att)), bilstm_att)],
        'bilstmTransitFocus': round(bl_focus, 4),
        'flux400':     [round(float(v), 6) for v in flux400],
        'xgbConf':     round(float(model_preds.get('XGBoost', {}).get('prob', 0.5)), 4),
        'cnnConf':     round(cnn_prob, 4),
        'bilstmConf':  round(bilstm_prob, 4),
        'nPoints':     int(SEQ_LEN),
        'n_train':     int(len(tr)),
        'n_val':       int(len(val)),
        'star_params':  star_params,
        # ── Per-star images ───────────────────────────────────────────────────
        'img_flux':    img_flux,
        'img_gradcam': img_gradcam,
        # ── Global images (same for all stars in this run) ────────────────────
        'img_preprocess':  preprocess_b64,
        'img_shap':        shap_summary_b64,
        'img_training':    training_curve_b64,
        'img_bilstm_training': bilstm_training_b64,
    }
    result = _san(result)

    fname = f"{RESULTS_DIR}/{sid.replace('/', '_').replace(' ', '_')}.json"
    with open(fname, 'w') as fout:
        json.dump(result, fout, indent=2)

    # Validate
    try:
        json.loads(open(fname).read())
        valid_tag = '✓ valid JSON'
    except Exception as e:
        valid_tag = f'✗ JSON ERROR: {e}'

    true_lbl_s = 'planet' if y[i] == 1 else 'fp'
    pred_lbl_s = 'planet' if pred_class == 1 else 'fp'
    flag_s = '  ⚠ MISMATCH' if true_lbl_s != pred_lbl_s else ''
    bilstm_lbl_s = 'planet' if bilstm_pred == 1 else 'fp'
    print(f'  [{sid}]  archive={true_lbl_s}  cnn={pred_lbl_s}  bilstm={bilstm_lbl_s}  '
          f'conf={cnn_prob:.3f}  cam={gc_loc:.3f}  bl_focus={bl_focus:.3f}  {valid_tag}{flag_s}')
    results.append(result)

print(f'\n✓ {len(results)} results saved to {RESULTS_DIR}/')
print('Next: Run Cell 9 for summary charts, then Cell 10 to download JSONs.')


---
## Cell 9 — Full Model Comparison + Explainability Summary

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

model_names = list(all_model_metrics.keys())
all_names   = model_names + ['1D CNN', 'BiLSTM']

# Tabular summary
print('\n' + '=' * 78)
print('  MULTI-MODEL PERFORMANCE COMPARISON')
print('=' * 78)
print('  {:<22}  {:>8}  {:>9}  {:>8}  {:>8}  {:>8}  {:>7}  {:>7}  {:>8}'.format(
      'Model','Accuracy','Precision','Recall','F1','AUC-ROC','PR-AUC','MCC','Bal.Acc'))
print('  ' + '-' * 96)
for mname in all_names:
    _dl_metrics = {'1D CNN': cnn_metrics, 'BiLSTM': bilstm_metrics}
    m   = _dl_metrics.get(mname) or all_model_metrics[mname]
    auc = m.get('auc', 0.5)
    print('  {:<22}  {:>8.4f}  {:>9.4f}  {:>8.4f}  {:>8.4f}  {:>8.4f}  {:>7.4f}  {:>7.4f}  {:>8.4f}'.format(
          mname, m['accuracy'], m['precision'], m['recall'], m['f1'], auc,
          m.get('pr_auc', 0.0), m.get('mcc', 0.0), m.get('bal_acc', 0.0)))

# Bar chart comparing all models
metrics_plot = ['accuracy', 'precision', 'recall', 'f1', 'auc', 'pr_auc', 'mcc', 'bal_acc']
x      = np.arange(len(metrics_plot))
width  = 0.8 / len(all_names)
colors = ['#64748b','#6366f1','#8b5cf6','#3b82f6','#0ea5e9','#f59e0b','#f97316','#10b981']

fig, ax = plt.subplots(figsize=(18, 5))
for j, mname in enumerate(all_names):
    _dl_metrics = {'1D CNN': cnn_metrics, 'BiLSTM': bilstm_metrics}
    m   = _dl_metrics.get(mname) or all_model_metrics[mname]
    vals = [m.get(k, 0.0) for k in metrics_plot]
    ax.bar(x + j*width - 0.4 + width/2, vals, width*0.9,
           label=mname, color=colors[j % len(colors)], alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(['Accuracy','Precision','Recall','F1','AUC-ROC','PR-AUC','MCC','Bal.Acc'], color='gray', fontsize=8)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score', color='gray')
ax.set_title('All Models — Performance Comparison', color='white')
ax.legend(facecolor='#1a2235', labelcolor='white', fontsize=9, ncol=4)
ax.set_facecolor('#111827'); fig.patch.set_facecolor('#0b0f1a')
ax.tick_params(colors='gray'); ax.spines[:].set_color('#1e2d47')
plt.tight_layout()
model_comparison_b64 = fig_to_b64(fig, dpi=90)
# Attach to all result JSONs
import glob as _glob
for _fp in _glob.glob(f'{RESULTS_DIR}/*.json'):
    _d = json.load(open(_fp)); _d['img_models'] = model_comparison_b64
    json.dump(_d, open(_fp,'w'), indent=2)
plt.savefig(f'{RESULTS_DIR}/model_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0b0f1a')
plt.show()
print(f'Chart saved to {RESULTS_DIR}/model_comparison.png')

# Explainability summary
print('\n' + '=' * 78)
print('  EXPLAINABILITY EVALUATION METRICS')
print('=' * 78)
em = explainability_metrics
print(f'  SHAP Faithfulness (acc drop masking top-3 features): {em["shap_faithfulness_drop"]:+.4f}')
print(f'    Positive = explanation correctly identifies the most important features')
print(f'  SHAP Sparsity (fraction of |SHAP| < 0.01):           {em["shap_sparsity"]:.4f}')
print(f'    Higher = model decisions driven by fewer, more distinct features')
print(f'  SHAP Consistency (Spearman rank corr across samples): {em["shap_consistency"]:.4f}')
print(f'    Near 1.0 = same features matter consistently across all light curves')
print(f'\n  Grad-CAM Localization (fraction of activation in transit window 0.45-0.55):')
for r in results:
    loc = r['explainability']['gradcam_localization']
    bar = chr(9608) * int(loc * 20)
    print(f'    {r["id"]:15s}  {loc:.3f}  {bar}')
print('    (1.0 = CNN perfectly focuses on the transit region)')

print(f'\n  BiLSTM Attention Localization (fraction of attention in transit window 0.45-0.55):')
for r in results:
    loc = r['explainability'].get('bilstm_localization', 0.0)
    bar = chr(9608) * int(loc * 20)
    print(f'    {r["id"]:15s}  {loc:.3f}  {bar}')
print('    (1.0 = BiLSTM attention perfectly focuses on the transit region)')



--- ## 📊 Cell 9b — SHAP Global Analysis + Dependence Plots (§14)

Global SHAP summary, feature importance ranking, and dependence plots for the top 3 features. Compares correct vs incorrect predictions.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHAP GLOBAL ANALYSIS — summary + dependence plots (§14)
# ══════════════════════════════════════════════════════════════════════════════
import shap, matplotlib.pyplot as plt
import numpy as np

# Use XGBoost (best supported by SHAP TreeExplainer)
try:
    _xgb = model_objects['XGBoost'][0]
    _explainer = shap.TreeExplainer(_xgb)

    # Compute SHAP values on full val+test set
    _X_eval = X_feat[val] if len(test) == 0 else np.vstack([X_feat[val], X_feat[test]])
    _y_eval = y[val]      if len(test) == 0 else np.concatenate([y[val], y[test]])
    _sv_raw = _explainer.shap_values(_X_eval)
    _sv = _sv_raw[1] if isinstance(_sv_raw, list) and len(_sv_raw) > 1 else _sv_raw

    # ── 1. Global bar plot (mean |SHAP|) ────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plt.sca(axes[0])
    shap.summary_plot(_sv, _X_eval, feature_names=feat_names,
                      plot_type='bar', show=False, max_display=15)
    axes[0].set_title('Global SHAP Feature Importance (mean |SHAP|)', fontsize=12)

    # ── 2. Beeswarm plot ─────────────────────────────────────────────────────
    plt.sca(axes[1])
    shap.summary_plot(_sv, _X_eval, feature_names=feat_names,
                      show=False, max_display=15)
    axes[1].set_title('SHAP Beeswarm (value distribution)', fontsize=12)
    plt.tight_layout()
    plt.savefig('/content/shap_global.png', dpi=100, bbox_inches='tight')
    plt.show(); plt.close()

    # ── 3. Dependence plots for top 3 features ───────────────────────────────
    _mean_abs = np.abs(_sv).mean(axis=0)
    _top3_idx = np.argsort(_mean_abs)[::-1][:3]
    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4))
    for ax, fi in zip(axes2, _top3_idx):
        shap.dependence_plot(fi, _sv, _X_eval, feature_names=feat_names,
                             ax=ax, show=False)
        ax.set_title(f'Dependence: {feat_names[fi]}', fontsize=10)
    plt.tight_layout()
    plt.savefig('/content/shap_dependence.png', dpi=100, bbox_inches='tight')
    plt.show(); plt.close()

    # ── 4. Correct vs incorrect SHAP comparison ──────────────────────────────
    _preds_eval = (model_objects['XGBoost'][2] if len(test) == 0
                   else np.concatenate([model_objects['XGBoost'][2],
                                        (model_objects['XGBoost'][0].predict_proba(X_feat[test])[:,1]
                                         >= model_thresholds.get('XGBoost', 0.5)).astype(int)]))
    _correct = _preds_eval == _y_eval
    if _correct.sum() > 0 and (~_correct).sum() > 0:
        fig3, axes3 = plt.subplots(1, 2, figsize=(14, 4))
        shap.summary_plot(_sv[_correct],  _X_eval[_correct],
                          feature_names=feat_names, show=False, ax=axes3[0], max_display=10)
        axes3[0].set_title(f'Correct predictions (n={_correct.sum()})', fontsize=11)
        shap.summary_plot(_sv[~_correct], _X_eval[~_correct],
                          feature_names=feat_names, show=False, ax=axes3[1], max_display=10)
        axes3[1].set_title(f'Misclassified (n={(~_correct).sum()})', fontsize=11)
        plt.tight_layout()
        plt.savefig('/content/shap_correct_vs_error.png', dpi=100, bbox_inches='tight')
        plt.show(); plt.close()
        print(f'\nSHAP comparison: {_correct.sum()} correct vs {(~_correct).sum()} misclassified')
    else:
        print('(All predictions correct — no misclassification SHAP comparison)')

    print('SHAP global analysis complete. Plots saved to /content/')
except Exception as e:
    print(f'SHAP analysis failed: {e}')


--- ## 🌌 Cell 9c — Cross-Mission Evaluation (§13)

Tests whether a model trained on Kepler generalises to TESS, and vice versa. Requires both missions in the dataset.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CROSS-MISSION EVALUATION — Train Kepler → Test TESS (§12, §13)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import f1_score as _f1s, roc_auc_score as _aucs, balanced_accuracy_score as _bas

missions_arr = np.array(all_missions[:len(y)])
kepler_idx = np.where(missions_arr == 'Kepler')[0]
tess_idx   = np.where(missions_arr == 'TESS')[0]

print(f"Kepler samples: {len(kepler_idx)} | TESS samples: {len(tess_idx)}")

if len(kepler_idx) < 5 or len(tess_idx) < 3:
    print("\n⚠ Insufficient samples for cross-mission evaluation.")
    print("  Add more TESS targets (TIC_IDS in Cell 3) and re-run.")
else:
    print("\n══ Cross-Mission Evaluation ══")
    print("  Train: Kepler  →  Test: TESS")
    print(f"  {'Model':<25}  {'F1':>8}  {'AUC':>8}  {'BalAcc':>8}")
    print("  " + "-"*55)

    X_kep = X_feat[kepler_idx]; y_kep = y[kepler_idx]
    X_tes = X_feat[tess_idx];   y_tes = y[tess_idx]

    cross_results = {}
    for mname, (clf_obj, _, _, _) in model_objects.items():
        if mname in ('Stacking Ensemble', 'Ensemble (Voting)'):
            continue   # skip meta-learners for cross-mission (refitting needed)
        try:
            from sklearn.base import clone as _clone
            from sklearn.preprocessing import RobustScaler as _RS
            _clf = _clone(clf_obj)
            _sc  = _RS()
            _clf.fit(_sc.fit_transform(X_kep), y_kep)
            _pt  = _clf.predict_proba(_sc.transform(X_tes))[:,1]
            _pd  = (_pt >= 0.5).astype(int)
            try:    _a = float(_aucs(y_tes, _pt))
            except: _a = 0.5
            _f  = float(_f1s(y_tes, _pd, zero_division=0))
            _ba = float(_bas(y_tes, _pd))
            cross_results[mname] = {'f1': _f, 'auc': _a, 'bal_acc': _ba}
            print(f"  {mname:<25}  {_f:>8.4f}  {_a:>8.4f}  {_ba:>8.4f}")
        except Exception as e:
            print(f"  {mname}: {e}")

    print("\nCross-mission results in cross_results dict.")
    print("Low performance here (vs in-mission) indicates limited cross-mission generalisation.")


---
## 📥 Cell 10 — Download JSON files to your computer

Run this to download all result JSONs. Then drag them into the **Results JSON** tab of the HTML prototype.

In [ ]:
from google.colab import files
import glob, os, zipfile

# Package ALL JSON results into one zip and download
json_files = sorted(glob.glob(f'{RESULTS_DIR}/*.json'))
print(f'Found {len(json_files)} JSON files')

zip_path = '/content/exoplanet_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in json_files:
        zf.write(fp, os.path.basename(fp))
        print(f'  + {os.path.basename(fp)}')

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zipped {len(json_files)} JSONs  ({size_mb:.1f} MB)')
files.download(zip_path)
print('Downloaded: exoplanet_results.zip  -- unzip and drag all .json files into the prototype sidebar')


---
## 📥 Cell 11 — (Optional) Download ALL result files as a zip

Downloads everything: JSON, Grad-CAM PNGs, comparison chart, training curve.

In [ ]:
import zipfile, os, glob
from google.colab import files

# Full archive: all JSONs + PNGs + model file
zip_path = '/content/exoplanet_full_archive.zip'
all_files = (
    sorted(glob.glob(f'{RESULTS_DIR}/*.json')) +
    sorted(glob.glob(f'{RESULTS_DIR}/*.png'))  +
    sorted(glob.glob(f'{MODELS_DIR}/*.json'))
)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in all_files:
        arcname = os.path.relpath(fp, '/content/')
        zf.write(fp, arcname)

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Full archive: {len(all_files)} files  ({size_mb:.1f} MB)')
files.download(zip_path)
print('Downloaded: exoplanet_full_archive.zip')
